# Diet Nitrate/Nitrite/Nitroso Oral Ecology In HPP

Focused analysis notebook for Paper 2: KG-enhanced dietary nitrate, nitrite, nitroso, and arginine/NO axes as distinct oral microbiome ecology exposures in HPP.

Main question: do chemically related but food-source-distinct nitrogen/NO dietary axes map to different oral nitrate-cycle microbial states?


## Analysis Design

- Main exposures: overall nitrate, vegetable nitrate, overall nitrite, processed nitrite, nitroso axis, and arginine/NO axis.
- Main oral ecology outcomes are prespecified nitrate-cycle taxa/features only, keeping the FDR denominator small.
- Main adjusted models include age, sex, BMI, smoking, alcohol, and diet-logging covariates.
- Age and sex main effects on oral ecology are tested separately.
- Processed-nitrite to nitroso-axis pathway analysis asks whether processed nitrite associations attenuate after adding the nitroso/nitrosamine dietary axis.
- Age/sex mediation-style analyses test whether age/sex-associated diet differences explain oral ecology differences.
- Supplementary analysis repeats adjusted exposure tests across all species-level MetaPhlAn features with a large FDR denominator.


## Methods Writing Guide For Paper 2

Use this notebook for: **Agentic diet-data enhancement and hypothesis discovery reveal divergent oral microbiome signatures of nitrate, nitrite, nitroso and arginine/NO dietary axes in HPP.**

Main claim should be food-source and chemical-context specificity, not age or sex interaction. Age and sex are tested as demographic correlates and mediation-style secondary analyses.


The processed-nitrite/nitroso pathway section is an observational mechanistic check: it tests whether the processed nitrite signal is statistically carried by the nitroso-axis exposure reconstruction, without claiming measured N-nitroso compounds or causal mediation.


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# If root detection fails inside TRE, set manually and rerun this cell.
# MANUAL_PROJECT_ROOT = Path("/home/ec2-user/studies/Diet_Data_Enhancement_Project")
MANUAL_PROJECT_ROOT = None


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = []
    for env_var in ["DIET_DATA_ENHANCEMENT_ROOT", "PROJECT_ROOT"]:
        if os.environ.get(env_var):
            candidates.append(Path(os.environ[env_var]).expanduser())
    if MANUAL_PROJECT_ROOT is not None:
        candidates.append(Path(MANUAL_PROJECT_ROOT).expanduser())
    candidates.extend([start, *start.parents])
    candidates.extend([
        Path.home() / "studies" / "Diet_Data_Enhancement_Project",
        Path.home() / "studies" / "Diet_Data_enhancement",
        Path.home() / "Diet_Data_Enhancement_Project",
        Path.home() / "Diet_Data_enhancement",
    ])
    seen = set()
    for candidate in candidates:
        if str(candidate) in seen:
            continue
        seen.add(str(candidate))
        if (candidate / "outputs" / "visualizations" / "denovo_hpp_kg_mega_flexible_data.js").exists():
            return candidate
        if (candidate / "outputs" / "enhanced_hpp").exists() and (candidate / "downstream_analysis").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "downstream_analysis" / "manual" / "nitrate" / "outputs" / "Diet_Nitrate_oral_ecology"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUTPUT_DIR =", OUTPUT_DIR)


In [ ]:
# Configuration
ID_COL = "participant_id"
FOOD_COL = "food_id"
REF_FOOD_COL = "hpp_food_id"
GRAMS_COL = "weight_g"

# Full mega KG exposure discovery.
KG_MODE = "full_mega"
KG_MAX_HOPS = 4
MEGA_KG_DATA_PATH = PROJECT_ROOT / "outputs" / "visualizations" / "denovo_hpp_kg_mega_flexible_data.js"
FOOD_FEATURE_TABLE = PROJECT_ROOT / "outputs" / "enhanced_hpp" / "1.denovo" / "hpp_feature_matrix_per_100g.csv"

# Exposure normalization.
# Primary metric reduces bias from people who log more often.
PRIMARY_EXPOSURE_METRIC = "exposure_per_1000g_logged"
# Secondary metric preserves body-relevant amount among days that were logged.
SECONDARY_EXPOSURE_METRIC = "exposure_g_per_logging_day"
MIN_LOGGED_FOOD_EVENTS = 5
MIN_TOTAL_LOGGED_G = 250

# Primary paper analysis excludes sparse/irregular diet loggers based on daily logging regularity.
MIN_RELIABLE_LOGGED_DAYS = 5
MIN_USABLE_LOGGED_DAYS = 3
MIN_RELIABLE_TOTAL_LOGGED_G = 500
MIN_RELIABLE_MEDIAN_DAILY_G = 500
MAX_RELIABLE_CV_DAILY_G = 1.5
MAX_RELIABLE_GAP_DAYS = 14
PRIMARY_LOGGING_QUALITY = "reliable"

FORCE_REBUILD_DIET_EXPOSURES = False
FORCE_REBUILD_ORAL_FEATURES = False
FORCE_REBUILD_CONFOUNDERS = False
CACHE_DIR = OUTPUT_DIR / "cache_pickle"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATHS = {
    "diet_events": CACHE_DIR / "diet_events.pkl",
    "food_features": CACHE_DIR / "food_features.pkl",
    "kg_nodes": CACHE_DIR / "kg_nodes.pkl",
    "kg_edges": CACHE_DIR / "kg_edges.pkl",
    "kg_reverse_adj": CACHE_DIR / "kg_reverse_adj.pkl",
    "exposure_tables": CACHE_DIR / "exposure_tables_oral_ecology_v1.pkl",
    "exposure_summary": CACHE_DIR / "exposure_summary_oral_ecology_v1.pkl",
    "oral_features": CACHE_DIR / "oral_features_relevant_taxa_v1.pkl",
    "confounders": CACHE_DIR / "confounders_lifestyle_v1.pkl",
}

EXPOSURE_SPECS = {
    "overall_nitrate": {
        "terms": ["nitrate", "non metal nitrates", "sodium nitrate", "potassium nitrate"],
        "source_note": "Broad KG-connected nitrate axis across all food sources.",
    },
    "vegetable_nitrate": {
        "terms": ["nitrate", "non metal nitrates"],
        "include_food_name_tokens": ["vegetable", "leafy", "spinach", "lettuce", "beet", "beetroot", "arugula", "rocket", "celery", "cabbage", "chard", "radish"],
        "exclude_food_name_tokens": ["smoked", "cured", "sausage", "bacon", "ham", "salami", "hot dog", "processed"],
        "source_note": "Vegetable-enriched nitrate axis, excluding obvious cured/smoked processed terms.",
    },
    "overall_nitrite": {
        "terms": ["nitrite", "non metal nitrites", "sodium nitrite", "potassium nitrite"],
        "source_note": "Broad KG-connected nitrite axis across all food sources.",
    },
    "processed_nitrite": {
        "terms": ["nitrite", "sodium nitrite", "sodium nitrate", "non metal nitrites"],
        "include_food_name_tokens": ["smoked", "cured", "sausage", "bacon", "ham", "salami", "hot dog", "processed"],
        "source_note": "Processed/cured nitrate-nitrite signal; contrast exposure for food matrix/source effects.",
    },
    "nitroso_axis": {
        "terms": ["nitroso", "nitrosamine", "n nitroso", "nitroso compound"],
        "source_note": "Exploratory downstream nitroso/nitrosamine axis.",
    },
    "arginine_no_axis": {
        "terms": ["arginine", "citrulline", "ornithine", "arginine and proline metabolism", "nitric oxide"],
        "source_note": "Host nitric-oxide substrate axis.",
    },
}

GROUP_ORDER = ["low", "mid", "high"]
GROUP_COLORS = {"low": "#2166ac", "mid": "#f4a582", "high": "#b2182b", "bottom_10": "#2166ac", "top_10": "#b2182b"}
MIN_GROUP_N = 30
FDR_ALPHA = 0.10

RELEVANT_TAXA_PATTERNS = {
    "neisseria": r"neisseria|neisseriaceae",
    "rothia": r"rothia",
    "rothia_dentocariosa": r"rothia.*dentocariosa|dentocariosa",
    "prevotella": r"prevotella|prevotellaceae",
    "veillonella": r"veillonella",
    "megasphaera": r"megasphaera",
    "moraxellaceae": r"moraxellaceae|moraxella",
    "burkholderiaceae": r"burkholderiaceae|burkholderia",
}

NITROGEN_PATHWAY_RE = re.compile(r"nitrate|nitrite|nitric|nitros|denitrification|nitrogen|\\bnar[A-Z]?\\b|\\bnir[A-Z]?\\b|\\bnor[A-Z]?\\b|\\bnos[Z]?\\b", re.I)

# Lifestyle fields based on HPP lifestyle_and_environment knowledgebase.
SMOKING_COLUMNS = ["smoking_current_status", "smoking_past_frequency", "smoking_total_above_100", "smoking_status", "smoking_current", "smoke"]
ALCOHOL_COLUMNS = [
    "alcohol_current_frequency", "alcohol_past_status",
    "alcohol_weekly_red_intake_number", "alcohol_weekly_white_champagne_intake_number",
    "alcohol_weekly_beer_cider_intake_number", "alcohol_weekly_spirits_intake_number",
    "alcohol_weekly_fortified_wine_intake_number", "alcohol_weekly_other_intake_number",
    "alcohol_monthly_red_intake_number", "alcohol_monthly_white_champagne_intake_number",
    "alcohol_monthly_beer_cider_intake_number", "alcohol_monthly_spirits_intake_number",
    "alcohol_monthly_fortified_wine_intake_number", "alcohol_monthly_other_intake_number",
    "alcohol_current_with_meals", "alcohol", "drinking_frequency",
]

BASE_COVARIATES = ["age", "sex", "bmi", "smoking_current_status", "smoking_total_above_100", "alcohol_current_frequency"]
LOGGING_COVARIATES = ["food_events", "logging_days", "total_logged_g"]


In [ ]:
def flatten_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if isinstance(out.index, pd.MultiIndex) or out.index.name is not None:
        out = out.reset_index()
    return out.loc[:, ~out.columns.duplicated()].copy()


def make_unique_columns(columns) -> list[str]:
    seen, unique = {}, []
    for col in columns:
        base = str(col)
        n = seen.get(base, 0)
        unique.append(base if n == 0 else f"{base}__dup{n}")
        seen[base] = n + 1
    return unique


def normalize_ids(df: pd.DataFrame, id_col: str = ID_COL) -> pd.DataFrame:
    out = flatten_index(df)
    out.columns = make_unique_columns(out.columns)
    if id_col not in out.columns:
        for alias in ["research_stage_id", "user_id", "RegistrationCode", "participant", "sample_id", "SampleID"]:
            if alias in out.columns:
                out = out.rename(columns={alias: id_col})
                break
    if id_col not in out.columns:
        raise ValueError(f"Could not find participant id column. Columns include: {out.columns.tolist()[:40]}")
    out[id_col] = out[id_col].astype(str)
    return out


def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, low_memory=False)
    if path.suffix.lower() in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="\t", low_memory=False)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() in {".arrow", ".feather"}:
        try:
            return pd.read_feather(path)
        except Exception:
            import pyarrow.feather as feather
            return feather.read_feather(path).to_pandas()
    raise ValueError(f"Unsupported table: {path}")


def normalize_token_text(text: object) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def token_contains(text: object, term: object) -> bool:
    clean_text = normalize_token_text(text)
    clean_term = normalize_token_text(term)
    if not clean_text or not clean_term:
        return False
    text_tokens = clean_text.split()
    term_tokens = clean_term.split()
    if len(term_tokens) == 1:
        return term_tokens[0] in text_tokens
    n = len(term_tokens)
    return any(text_tokens[i:i+n] == term_tokens for i in range(len(text_tokens)-n+1))


def contains_any(text: object, terms: list[str]) -> bool:
    return any(token_contains(text, term) for term in terms)


def finite_numeric(s: pd.Series) -> np.ndarray:
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=float)


def p_adjust_bh(p_values) -> np.ndarray:
    p = np.asarray(pd.to_numeric(pd.Series(p_values), errors="coerce"), dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return q
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    n = len(ranked)
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    out = np.empty(n)
    out[order] = adjusted
    q[valid] = out
    return q


def cohens_d(x, y) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    if len(x) < 2 or len(y) < 2:
        return np.nan
    pooled = math.sqrt(((len(x)-1)*np.var(x, ddof=1) + (len(y)-1)*np.var(y, ddof=1)) / (len(x)+len(y)-2))
    if pooled == 0:
        return np.nan
    return (np.mean(y) - np.mean(x)) / pooled


def cliffs_delta(x, y) -> float:
    """Second group minus first group, matching cohens_d(x, y)."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    if len(x) == 0 or len(y) == 0:
        return np.nan
    ranks = stats.rankdata(np.concatenate([x, y]))
    rx = ranks[:len(x)].sum()
    u_x = rx - len(x) * (len(x) + 1) / 2
    delta_x_minus_y = (2 * u_x / (len(x) * len(y))) - 1
    return -delta_x_minus_y


def cohens_d_ci(d: float, n1: int, n2: int, z: float = 1.96) -> tuple[float, float]:
    if not np.isfinite(d) or n1 < 2 or n2 < 2:
        return np.nan, np.nan
    se = math.sqrt((n1 + n2) / (n1 * n2) + (d**2) / (2 * (n1 + n2 - 2)))
    return d - z * se, d + z * se


def format_p(p):
    if p is None or not np.isfinite(p):
        return "p=NA"
    return f"p={p:.1e}" if p < 1e-4 else f"p={p:.4f}"


def pretty_label(value):
    return str(value).replace("_", " ")


## 1. Diet Data And KG-Derived Exposure Construction


In [ ]:
def progress(message: str, start_time: float | None = None):
    if start_time is None:
        print(message, flush=True)
    else:
        print(f"{message} elapsed={time.time() - start_time:.1f}s", flush=True)


def load_diet_events_from_phenoloader() -> pd.DataFrame:
    from pheno_utils import PhenoLoader
    from pheno_utils.config import DATASETS_PATH

    t0 = time.time()
    progress("[diet 1/7] Initializing PhenoLoader('diet_logging')...")
    pl = PhenoLoader("diet_logging", age_sex_dataset=None, errors="warn")
    dataset_dir = Path(DATASETS_PATH) / pl.dataset
    candidate_paths = [dataset_dir / "diet_logging_events.parquet"]
    progress(f"[diet 1/7] PhenoLoader initialized; dataset_dir={dataset_dir}", t0)

    if "diet_logging_events" in getattr(pl, "dfs", {}):
        progress("[diet 2/7] Using diet_logging_events already loaded by PhenoLoader...")
        out = normalize_ids(flatten_index(pl.dfs["diet_logging_events"]))
        progress(f"[diet 2/7] diet_events shape={out.shape}", t0)
        return out

    if "diet_logging" in getattr(pl, "dfs", {}):
        summary_df = flatten_index(pl.dfs["diet_logging"])
        if "diet_logging_events" in summary_df.columns:
            for rel_path in summary_df["diet_logging_events"].dropna().astype(str).unique()[:20]:
                candidate_paths.append(dataset_dir / rel_path)

    events_path = next((path for path in candidate_paths if path.exists()), None)
    if events_path is None:
        raise FileNotFoundError("Could not load diet events via PhenoLoader or dataset path.")

    progress(f"[diet 2/7] Reading diet events parquet: {events_path}")
    progress("[diet 2/7] Trying column-only read first; this is much faster if parquet metadata supports it...")
    candidate_column_sets = []
    id_candidates = [ID_COL, "participant_id", "participant" ]
    food_candidates = [FOOD_COL, "hpp_food_id", "food_id"]
    grams_candidates = [GRAMS_COL, "grams_consumed", "amount_g", "food_weight", "weight_g"]
    date_candidates = ["event_date", "date", "meal_date", "logging_date", "collection_date", "timestamp", "datetime", "start_time", "created_at", "meal_time"]
    for id_col in id_candidates:
        for food_col in food_candidates:
            for grams_col in grams_candidates:
                for date_col in date_candidates:
                    candidate_column_sets.append([id_col, food_col, grams_col, date_col])

    errors = []
    diet = None
    for cols in candidate_column_sets:
        try:
            diet = pd.read_parquet(events_path, columns=list(dict.fromkeys(cols)))
            progress(f"[diet 2/7] Column-only read worked with columns={cols}; shape={diet.shape}", t0)
            break
        except Exception as exc:
            errors.append(str(exc)[:160])
    if diet is None:
        progress("[diet 2/7] Column-only read did not match the file schema; falling back to full parquet read...")
        diet = pd.read_parquet(events_path)
        progress(f"[diet 2/7] Full diet_events read complete; shape={diet.shape}", t0)

    progress("[diet 2/7] Normalizing diet event identifiers...")
    diet = normalize_ids(flatten_index(diet))
    progress(f"[diet 2/7] Diet events ready; shape={diet.shape}", t0)
    return diet


def js_value_after_key(text: str, key: str):
    marker = f"{key}:"
    start = text.index(marker) + len(marker)
    decoder = json.JSONDecoder()
    value, _end = decoder.raw_decode(text[start:])
    return value


def load_mega_kg_data(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not path.exists():
        raise FileNotFoundError(f"Mega KG data file not found: {path}")
    print("Loading full mega KG:", path, flush=True)
    text = path.read_text(encoding="utf-8")
    kinds = js_value_after_key(text, "kinds")
    relations = js_value_after_key(text, "relations")
    compact_nodes = js_value_after_key(text, "nodes")
    compact_edges = js_value_after_key(text, "edges")

    nodes = pd.DataFrame(compact_nodes, columns=["key", "label", "kind_idx", "x", "y"])
    nodes["kind"] = nodes["kind_idx"].map(lambda i: kinds[int(i)])
    nodes = nodes[["key", "kind", "label"]]

    key_by_idx = nodes["key"].astype(str).to_numpy()
    edges = pd.DataFrame(compact_edges, columns=["source_idx", "target_idx", "relation_idx"])
    edges["source"] = key_by_idx[edges["source_idx"].astype(int).to_numpy()]
    edges["target"] = key_by_idx[edges["target_idx"].astype(int).to_numpy()]
    edges["relation"] = edges["relation_idx"].map(lambda i: relations[int(i)])
    edges = edges[["source", "target", "relation"]]
    return nodes, edges


def build_reverse_adjacency(kg_edges: pd.DataFrame) -> dict[str, set[str]]:
    reverse_adj = {}
    for source, target in kg_edges[["source", "target"]].dropna().astype(str).itertuples(index=False):
        reverse_adj.setdefault(target, set()).add(source)
    return reverse_adj


def kg_target_nodes_for_terms(kg_nodes: pd.DataFrame, terms: list[str]) -> set[str]:
    target_nodes = set()
    search_cols = [c for c in ["key", "label", "kind"] if c in kg_nodes.columns]
    for row in kg_nodes[search_cols].fillna("").astype(str).itertuples(index=False, name=None):
        row_text = " | ".join(row)
        if any(token_contains(row_text, term) for term in terms):
            target_nodes.add(str(row[0]))
    return target_nodes


def hpp_food_id_from_node(node: str) -> str | None:
    node = str(node)
    return node.split(":", 1)[1] if node.startswith("hpp_food:") else None


def kg_path_connected_food_ids(kg_nodes, reverse_adj, terms, max_hops=KG_MAX_HOPS) -> tuple[set[str], dict]:
    target_nodes = kg_target_nodes_for_terms(kg_nodes, terms)
    frontier = set(target_nodes)
    visited = set(target_nodes)
    food_ids = set()
    for depth in range(max_hops + 1):
        for node in frontier:
            food_id = hpp_food_id_from_node(node)
            if food_id is not None:
                food_ids.add(food_id)
        if depth == max_hops:
            break
        next_frontier = set()
        for node in frontier:
            next_frontier.update(reverse_adj.get(node, set()))
        next_frontier -= visited
        visited.update(next_frontier)
        frontier = next_frontier
        if not frontier:
            break
    return food_ids, {"target_node_count": len(target_nodes), "visited_node_count": len(visited), "connected_food_count": len(food_ids), "max_hops": max_hops, "example_target_nodes": " | ".join(sorted(target_nodes)[:15])}


def load_food_features() -> pd.DataFrame:
    food = read_table(FOOD_FEATURE_TABLE)
    food[REF_FOOD_COL] = food[REF_FOOD_COL].astype(str)
    return food


def filter_connected_foods(food_ids: set[str], food_features: pd.DataFrame, spec: dict) -> set[str]:
    if not food_ids:
        return set()
    ref = food_features[food_features[REF_FOOD_COL].astype(str).isin({str(x) for x in food_ids})].copy()
    text_cols = [c for c in ["hpp_food_name", "hpp_product_name", "hpp_short_description", "hpp_category", "canonical_name", "canonical_category"] if c in ref.columns]
    if not text_cols:
        return {str(x) for x in food_ids}
    name_text = ref[text_cols].fillna("").astype(str).agg(" | ".join, axis=1)
    include = spec.get("include_food_name_tokens") or []
    exclude = spec.get("exclude_food_name_tokens") or []
    keep = pd.Series(True, index=ref.index)
    if include:
        keep &= name_text.map(lambda v: contains_any(v, include))
    if exclude:
        keep &= ~name_text.map(lambda v: contains_any(v, exclude))
    return set(ref.loc[keep, REF_FOOD_COL].astype(str))


In [ ]:
def infer_date_col(diet_events: pd.DataFrame) -> str | None:
    candidates = ["event_date", "date", "meal_date", "logging_date", "collection_date", "timestamp", "datetime", "start_time", "created_at", "meal_time"]
    for col in candidates:
        if col in diet_events.columns:
            return col
    for col in diet_events.columns:
        if re.search("date|time", str(col), re.I):
            return col
    return None


def max_gap_days_from_dates(date_series) -> float:
    dates = pd.to_datetime(pd.Series(sorted(pd.Series(date_series).dropna().unique())))
    if len(dates) < 2:
        return np.nan
    return float(dates.diff().dt.days.iloc[1:].max())


def prepare_diet_events_for_exposure(diet_events: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    t0 = time.time()
    progress("[diet 3/7] Preparing diet event columns for logging quality and exposures...")
    food_col = FOOD_COL if FOOD_COL in diet_events.columns else "hpp_food_id"
    grams_col = GRAMS_COL if GRAMS_COL in diet_events.columns else "grams_consumed"
    date_col = infer_date_col(diet_events)
    if date_col is None:
        raise ValueError(f"Could not infer date column from diet_events columns: {diet_events.columns.tolist()[:80]}")
    missing = [c for c in [ID_COL, food_col, grams_col, date_col] if c not in diet_events.columns]
    if missing:
        raise ValueError(f"Diet event table is missing required columns after loading: {missing}. Available columns start: {diet_events.columns.tolist()[:40]}")

    d = normalize_ids(flatten_index(diet_events))[[ID_COL, food_col, grams_col, date_col]].copy()
    d = d.rename(columns={food_col: "_diet_food_id", grams_col: "_diet_grams", date_col: "_diet_date_raw"})
    d[ID_COL] = d[ID_COL].astype(str)
    d["_diet_food_id"] = d["_diet_food_id"].astype(str)
    d["_diet_grams"] = pd.to_numeric(d["_diet_grams"], errors="coerce").fillna(0).clip(lower=0)
    d["log_date"] = pd.to_datetime(d["_diet_date_raw"], errors="coerce").dt.date
    d["logging_date_norm"] = d["log_date"].astype(str)
    d.loc[d["logging_date_norm"].isin(["NaT", "None", "nan"]), "logging_date_norm"] = np.nan
    progress(f"[diet 3/7] Prepared diet event table shape={d.shape}; participants={d[ID_COL].nunique():,}; events={len(d):,}", t0)
    return d, {"food_col": food_col, "grams_col": grams_col, "date_col": date_col}


def build_logging_regularity_table(prepared_diet_events: pd.DataFrame, columns_info: dict) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    t0 = time.time()
    progress("[diet 4/7] Calculating daily logging regularity and sparse/irregular flags...")
    d = prepared_diet_events.dropna(subset=["log_date"]).copy()

    daily = d.groupby([ID_COL, "log_date"], as_index=False).agg(
        daily_total_g=("_diet_grams", "sum"),
        daily_food_events=("_diet_food_id", "count"),
    )
    progress(f"[diet 4/7] Daily table built; shape={daily.shape}; participant-days={len(daily):,}", t0)

    logging = daily.groupby(ID_COL).agg(
        first_log_date=("log_date", "min"),
        last_log_date=("log_date", "max"),
        logged_days=("log_date", "nunique"),
        total_logged_g_from_daily=("daily_total_g", "sum"),
        median_daily_g=("daily_total_g", "median"),
        mean_daily_g=("daily_total_g", "mean"),
        sd_daily_g=("daily_total_g", "std"),
        median_events_per_day=("daily_food_events", "median"),
        mean_events_per_day=("daily_food_events", "mean"),
        max_gap_days=("log_date", max_gap_days_from_dates),
    ).reset_index()

    logging["observed_span_days"] = (pd.to_datetime(logging["last_log_date"]) - pd.to_datetime(logging["first_log_date"])).dt.days + 1
    logging["logging_coverage"] = logging["logged_days"] / logging["observed_span_days"].replace(0, np.nan)
    logging["cv_daily_g"] = logging["sd_daily_g"] / logging["mean_daily_g"].replace(0, np.nan)

    logging["flag_too_few_days"] = logging["logged_days"] < MIN_USABLE_LOGGED_DAYS
    logging["flag_low_total_g"] = logging["total_logged_g_from_daily"] < MIN_RELIABLE_TOTAL_LOGGED_G
    logging["flag_low_daily_g"] = logging["median_daily_g"] < MIN_RELIABLE_MEDIAN_DAILY_G
    logging["flag_high_variability"] = logging["cv_daily_g"] > MAX_RELIABLE_CV_DAILY_G
    logging["flag_large_gap"] = logging["max_gap_days"].fillna(0) > MAX_RELIABLE_GAP_DAYS

    logging["logging_quality"] = "reliable"
    logging.loc[
        (logging["logged_days"] < MIN_RELIABLE_LOGGED_DAYS)
        | logging["flag_low_total_g"]
        | logging["flag_low_daily_g"]
        | logging["flag_high_variability"]
        | logging["flag_large_gap"],
        "logging_quality",
    ] = "sparse_or_irregular"
    logging.loc[logging["flag_too_few_days"] | logging["flag_low_total_g"], "logging_quality"] = "exclude_candidate"
    logging["analysis_weight_by_days"] = np.minimum(1.0, logging["logged_days"] / MIN_RELIABLE_LOGGED_DAYS)
    flag_cols = ["flag_too_few_days", "flag_low_total_g", "flag_low_daily_g", "flag_high_variability", "flag_large_gap"]
    logging["n_quality_flags"] = logging[flag_cols].sum(axis=1)

    thresholds = {
        "date_col": columns_info["date_col"],
        "grams_col": columns_info["grams_col"],
        "food_col": columns_info["food_col"],
        "min_reliable_logged_days": MIN_RELIABLE_LOGGED_DAYS,
        "min_usable_logged_days": MIN_USABLE_LOGGED_DAYS,
        "min_reliable_total_logged_g": MIN_RELIABLE_TOTAL_LOGGED_G,
        "min_reliable_median_daily_g": MIN_RELIABLE_MEDIAN_DAILY_G,
        "max_reliable_cv_daily_g": MAX_RELIABLE_CV_DAILY_G,
        "max_reliable_gap_days": MAX_RELIABLE_GAP_DAYS,
        "primary_logging_quality": PRIMARY_LOGGING_QUALITY,
    }
    progress(f"[diet 4/7] Logging quality table complete; participants={logging[ID_COL].nunique():,}", t0)
    return logging, daily, thresholds


def participant_exposure_from_kg_connected_foods(prepared_diet_events: pd.DataFrame, connected_food_ids: set[str], exposure_name: str, logging_quality: pd.DataFrame | None = None) -> pd.DataFrame:
    t0 = time.time()
    connected = {str(x) for x in connected_food_ids}
    progress(f"[diet 7/7] Aggregating participant exposure for {exposure_name}; connected foods={len(connected):,}...")
    d = prepared_diet_events[[ID_COL, "_diet_food_id", "_diet_grams", "logging_date_norm"]].copy()
    d["is_connected_food"] = d["_diet_food_id"].isin(connected)
    d["absolute_exposure_g"] = np.where(d["is_connected_food"], d["_diet_grams"], 0.0)

    out = d.groupby(ID_COL, as_index=False).agg(
        absolute_exposure_g=("absolute_exposure_g", "sum"),
        connected_food_events=("is_connected_food", "sum"),
        food_events=("_diet_food_id", "count"),
        unique_foods=("_diet_food_id", "nunique"),
        total_logged_g=("_diet_grams", "sum"),
        logging_days=("logging_date_norm", lambda s: s.dropna().nunique()),
    )
    out["logging_days"] = out["logging_days"].replace(0, np.nan)
    out["exposure_per_1000g_logged"] = np.where(out["total_logged_g"] > 0, out["absolute_exposure_g"] / out["total_logged_g"] * 1000.0, np.nan)
    out["exposure_g_per_logging_day"] = out["absolute_exposure_g"] / out["logging_days"]
    out["logged_g_per_day"] = out["total_logged_g"] / out["logging_days"]
    out["log1p_absolute_exposure_g"] = np.log1p(out["absolute_exposure_g"])
    out["log1p_exposure_per_1000g_logged"] = np.log1p(out["exposure_per_1000g_logged"].fillna(0))
    out["log1p_exposure_g_per_logging_day"] = np.log1p(out["exposure_g_per_logging_day"].fillna(0))
    out["exposure"] = exposure_name

    if logging_quality is not None:
        keep_cols = [ID_COL, "logging_quality", "analysis_weight_by_days", "median_daily_g", "median_events_per_day", "max_gap_days", "cv_daily_g", "n_quality_flags", "flag_large_gap", "flag_low_daily_g"]
        out = out.merge(logging_quality[[c for c in keep_cols if c in logging_quality.columns]], on=ID_COL, how="left")
    else:
        out["logging_quality"] = "not_evaluated"

    out["exposure_value"] = out[PRIMARY_EXPOSURE_METRIC]
    out["log1p_exposure_value"] = np.log1p(out["exposure_value"].fillna(0))
    out["passes_logging_filter"] = out["logging_quality"].eq(PRIMARY_LOGGING_QUALITY)
    progress(f"[diet 7/7] {exposure_name} exposure table complete; shape={out.shape}; nonzero={(out['exposure_value'] > 0).sum():,}", t0)
    return out


def usable_metric(values: pd.Series, min_unique: int = 3) -> bool:
    v = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    return len(v) >= 3 * MIN_GROUP_N and v.nunique() >= min_unique and v.max() > v.min()


def choose_effective_exposure_metric(df: pd.DataFrame) -> str:
    candidates = [PRIMARY_EXPOSURE_METRIC, SECONDARY_EXPOSURE_METRIC, "absolute_exposure_g"]
    for metric in candidates:
        if metric in df.columns and usable_metric(df[metric]):
            return metric
    return PRIMARY_EXPOSURE_METRIC if PRIMARY_EXPOSURE_METRIC in df.columns else candidates[-1]


def assign_exposure_groups(values: pd.Series) -> pd.Series:
    v = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan)
    out = pd.Series(np.nan, index=v.index, dtype="object")
    ok = v.notna()
    if ok.sum() == 0:
        return out
    if v.loc[ok].nunique() < 3 or v.loc[ok].max() == v.loc[ok].min():
        out.loc[ok] = "low"
        return out
    ranks = v.loc[ok].rank(method="first")
    groups = pd.qcut(ranks, q=3, labels=GROUP_ORDER)
    out.loc[ok] = groups.astype(str).to_numpy()
    return out


def assign_decile_groups(values: pd.Series) -> pd.Series:
    v = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan)
    out = pd.Series(np.nan, index=v.index, dtype="object")
    ok = v.notna()
    if ok.sum() < 2 * MIN_GROUP_N or v.loc[ok].nunique() < 3:
        return out
    ranks = v.loc[ok].rank(method="first")
    lo, hi = ranks.quantile([0.10, 0.90]).to_numpy()
    out.loc[ok & (ranks <= lo)] = "bottom_10"
    out.loc[ok & (ranks >= hi)] = "top_10"
    return out


def finalize_exposure_table(exposure: pd.DataFrame, exposure_name: str) -> pd.DataFrame:
    exposure = exposure.copy()
    metric = choose_effective_exposure_metric(exposure)
    exposure["effective_exposure_metric"] = metric
    exposure["exposure_value"] = pd.to_numeric(exposure[metric], errors="coerce").fillna(0)
    exposure["log1p_exposure_value"] = np.log1p(exposure["exposure_value"].clip(lower=0))
    exposure["exposure_group"] = assign_exposure_groups(exposure["exposure_value"])
    exposure["decile_group"] = assign_decile_groups(exposure["exposure_value"])
    counts = exposure["exposure_group"].value_counts(dropna=False).to_dict()
    if any(counts.get(g, 0) < MIN_GROUP_N for g in GROUP_ORDER):
        print(f"WARNING: {exposure_name} has weak tertile groups using {metric}: {counts}", flush=True)
    else:
        print(f"{exposure_name}: using {metric} for groups; counts={counts}", flush=True)
    return exposure


def build_exposure_tables() -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    t0 = time.time()
    if CACHE_PATHS["exposure_tables"].exists() and CACHE_PATHS["exposure_summary"].exists() and not FORCE_REBUILD_DIET_EXPOSURES:
        progress("Loading cached reliable-logger exposure tables")
        cached_tables = pd.read_pickle(CACHE_PATHS["exposure_tables"])
        cached_tables = {name: finalize_exposure_table(df, name) for name, df in cached_tables.items()}
        return cached_tables, pd.read_pickle(CACHE_PATHS["exposure_summary"])

    progress("[diet 0/7] Building reliable-logger diet exposure tables from scratch...")
    diet_events = load_diet_events_from_phenoloader()
    progress("[diet 2/7] Caching raw/minimal diet events for later reuse...")
    diet_events.to_pickle(CACHE_PATHS["diet_events"])

    prepared_diet_events, columns_info = prepare_diet_events_for_exposure(diet_events)
    logging_quality, daily_logging, logging_thresholds = build_logging_regularity_table(prepared_diet_events, columns_info)
    pd.to_pickle(logging_quality, CACHE_DIR / "logging_quality_reliable_loggers_v1.pkl")
    pd.to_pickle(daily_logging, CACHE_DIR / "daily_logging_reliable_loggers_v1.pkl")
    pd.Series(logging_thresholds).to_json(CACHE_DIR / "logging_thresholds_reliable_loggers_v1.json", indent=2)

    print("Diet logging quality counts:")
    display(logging_quality["logging_quality"].value_counts(dropna=False).rename_axis("logging_quality").reset_index(name="participants"))
    print("Diet logging flag counts:")
    flag_cols = ["flag_too_few_days", "flag_low_total_g", "flag_low_daily_g", "flag_high_variability", "flag_large_gap"]
    display(logging_quality[flag_cols].sum().sort_values(ascending=False).rename_axis("flag").reset_index(name="participants"))

    progress("[diet 5/7] Loading enhanced food feature table...")
    food_features = load_food_features()
    food_features.to_pickle(CACHE_PATHS["food_features"])
    progress(f"[diet 5/7] Food feature table ready; shape={food_features.shape}", t0)

    progress("[diet 6/7] Loading full mega KG and reverse adjacency cache...")
    if CACHE_PATHS["kg_nodes"].exists() and CACHE_PATHS["kg_edges"].exists():
        kg_nodes = pd.read_pickle(CACHE_PATHS["kg_nodes"])
        kg_edges = pd.read_pickle(CACHE_PATHS["kg_edges"])
        progress(f"[diet 6/7] Loaded cached KG nodes={kg_nodes.shape}, edges={kg_edges.shape}", t0)
    else:
        kg_nodes, kg_edges = load_mega_kg_data(MEGA_KG_DATA_PATH)
        kg_nodes.to_pickle(CACHE_PATHS["kg_nodes"])
        kg_edges.to_pickle(CACHE_PATHS["kg_edges"])
        progress(f"[diet 6/7] Parsed and cached KG nodes={kg_nodes.shape}, edges={kg_edges.shape}", t0)

    if CACHE_PATHS["kg_reverse_adj"].exists():
        reverse_adj = pd.read_pickle(CACHE_PATHS["kg_reverse_adj"])
        progress(f"[diet 6/7] Loaded cached reverse adjacency; targets={len(reverse_adj):,}", t0)
    else:
        reverse_adj = build_reverse_adjacency(kg_edges)
        pd.to_pickle(reverse_adj, CACHE_PATHS["kg_reverse_adj"])
        progress(f"[diet 6/7] Built reverse adjacency; targets={len(reverse_adj):,}", t0)

    exposure_tables, rows = {}, []
    total_exposures = len(EXPOSURE_SPECS)
    for idx, (exposure_name, spec) in enumerate(EXPOSURE_SPECS.items(), start=1):
        step_t = time.time()
        progress(f"[diet 7/7] Exposure {idx}/{total_exposures}: finding KG-connected foods for {exposure_name}...")
        food_ids, kg_info = kg_path_connected_food_ids(kg_nodes, reverse_adj, spec["terms"])
        progress(f"[diet 7/7] {exposure_name}: KG returned {len(food_ids):,} food ids; applying food-name filters...")
        filtered_food_ids = filter_connected_foods(food_ids, food_features, spec)
        progress(f"[diet 7/7] {exposure_name}: {len(filtered_food_ids):,} foods after filters; aggregating participant exposures...")
        exposure_all = participant_exposure_from_kg_connected_foods(prepared_diet_events, filtered_food_ids, exposure_name, logging_quality=logging_quality)
        exposure = exposure_all[exposure_all["passes_logging_filter"]].copy()
        exposure = finalize_exposure_table(exposure, exposure_name)
        exposure_tables[exposure_name] = exposure
        rows.append({
            "exposure": exposure_name,
            "terms": ", ".join(spec["terms"]),
            "source_note": spec.get("source_note", ""),
            "connected_foods_after_filter": len(filtered_food_ids),
            "participants_before_logging_filter": exposure_all[ID_COL].nunique(),
            "participants_after_reliable_logging_filter": exposure[ID_COL].nunique(),
            "excluded_sparse_or_irregular": int(exposure_all["logging_quality"].ne(PRIMARY_LOGGING_QUALITY).sum()),
            "primary_logging_quality": PRIMARY_LOGGING_QUALITY,
            "primary_exposure_metric": PRIMARY_EXPOSURE_METRIC,
            "secondary_exposure_metric": SECONDARY_EXPOSURE_METRIC,
            "effective_exposure_metric": exposure["effective_exposure_metric"].iloc[0] if "effective_exposure_metric" in exposure.columns and len(exposure) else np.nan,
            **kg_info,
            **{f"logging_{k}": v for k, v in logging_thresholds.items()},
        })
        progress(f"[diet 7/7] Finished {exposure_name}; reliable participants={exposure[ID_COL].nunique():,}", step_t)
    summary = pd.DataFrame(rows)
    progress("[diet 7/7] Saving exposure table caches...")
    pd.to_pickle(exposure_tables, CACHE_PATHS["exposure_tables"])
    summary.to_pickle(CACHE_PATHS["exposure_summary"])
    progress("[diet 7/7] Exposure construction complete", t0)
    return exposure_tables, summary


exposure_tables, exposure_summary = build_exposure_tables()
display(exposure_summary)
for name, df in exposure_tables.items():
    print(name, df.shape, df["exposure_group"].value_counts(dropna=False).to_dict(), df["decile_group"].value_counts(dropna=False).to_dict())


## 1. Nitrate/Nitrite/Nitroso Oral Ecology: Preprocessing


In [ ]:
ORAL_METADATA_COLS = {ID_COL, "cohort", "research_stage", "array_index", "collection_date"}
ORAL_BULK_SPECS = [
    ("oral_metaphlan_genus", r"^metaphlan_abundance_genus_parquet$", "taxa"),
    ("oral_metaphlan_species", r"^metaphlan_abundance_species_parquet$", "taxa"),
    ("oral_metaphlan_family", r"^metaphlan_abundance_family_parquet$", "taxa"),
    ("oral_humann_pathway_abundance_pathway_level", r"^humann_aggregated_pathway_abundance_pathway_level_arrow$|^humann_pathway_abundance_pathway_level_parquet$", "pathway"),
    ("oral_humann_pathway_coverage_pathway_level", r"^humann_aggregated_pathway_coverage_pathway_level_arrow$|^humann_pathway_coverage_pathway_level_parquet$", "pathway"),
]


def make_phenoloader(dataset: str):
    from pheno_utils import PhenoLoader
    return PhenoLoader(dataset, errors="warn")


def phenoloader_dataset_dir(pl) -> Path | None:
    try:
        from pheno_utils.config import DATASETS_PATH
        return Path(DATASETS_PATH) / pl.dataset
    except Exception as exc:
        print("Could not infer DATASETS_PATH:", exc)
        return None


def load_pheno_main_table(dataset: str, table: str | None = None, return_loader: bool = False):
    pl = make_phenoloader(dataset)
    table = table or dataset
    if table in getattr(pl, "dfs", {}):
        df = normalize_ids(flatten_index(pl.dfs[table]))
    else:
        df = normalize_ids(flatten_index(pl[table]))
    return (df, pl) if return_loader else df


def candidate_bulk_paths(raw: str, dataset_dir: Path | None) -> list[Path]:
    raw = str(raw).strip()
    if not raw or raw.lower() in {"nan", "none"} or raw.startswith("s3://"):
        return []
    cleaned = raw.lstrip("./")
    candidates = [Path(raw)]
    if dataset_dir is not None:
        candidates.extend([dataset_dir / raw, dataset_dir / cleaned, dataset_dir / "oral_microbiome" / raw, dataset_dir / "oral_microbiome" / cleaned])
    return list(dict.fromkeys(candidates))


def load_optional_bulk_table(primary: pd.DataFrame, path_regex: str, dataset_dir: Path | None):
    rx = re.compile(path_regex, re.I)
    path_cols = [c for c in primary.columns if rx.search(str(c))]
    if not path_cols:
        print("No oral bulk path column matched", path_regex)
        return None, None
    for col in path_cols:
        for raw in primary[col].dropna().astype(str).unique()[:20]:
            for candidate in candidate_bulk_paths(raw, dataset_dir):
                if not candidate.exists():
                    continue
                table = read_table(candidate)
                table = flatten_index(table)
                table.columns = make_unique_columns(table.columns)
                print(f"Loaded {col}: {candidate} shape={table.shape}")
                return table, candidate
    print("No readable local file for", path_regex)
    return None, None


def choose_participant_id_col(df: pd.DataFrame) -> str | None:
    for col in [ID_COL, "participant_id", "sample_id", "sample_name", "wgs_dna_code", "SampleID", "sample"]:
        if col in df.columns:
            return col
    return None


def sanitize_feature_name(value: object) -> str:
    text = str(value).replace("|", "_")
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()
    return text[:180] if text else "feature"


def normalize_wide_microbiome_table(df: pd.DataFrame, prefix: str) -> pd.DataFrame | None:
    df = flatten_index(df)
    df.columns = make_unique_columns(df.columns)
    id_col = choose_participant_id_col(df)
    if id_col is None:
        return None
    df = df.rename(columns={id_col: ID_COL})
    feature_cols = [c for c in df.columns if c not in ORAL_METADATA_COLS]
    numeric_parts = {}
    for c in feature_cols:
        vals = pd.to_numeric(df.loc[:, c], errors="coerce")
        if vals.notna().sum() > 0:
            numeric_parts[f"{prefix}_{sanitize_feature_name(c)}"] = vals.to_numpy()
    if not numeric_parts:
        return None
    out = pd.concat([pd.DataFrame({ID_COL: df[ID_COL].astype(str).to_numpy()}), pd.DataFrame(numeric_parts, index=df.index)], axis=1)
    out.columns = make_unique_columns(out.columns)
    return out.copy().groupby(ID_COL, as_index=False).mean(numeric_only=True)


def alpha_diversity_from_matrix(matrix: pd.DataFrame, prefix: str) -> pd.DataFrame:
    features = [c for c in matrix.select_dtypes(include="number").columns if c != ID_COL]
    x = matrix[features].clip(lower=0).fillna(0).to_numpy(dtype=float)
    row_sums = x.sum(axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.divide(x, row_sums[:, None], out=np.zeros_like(x), where=row_sums[:, None] > 0)
        logp = np.where(p > 0, np.log(p), 0)
    return pd.DataFrame({
        ID_COL: matrix[ID_COL].astype(str).to_numpy(),
        f"{prefix}_shannon": -(p * logp).sum(axis=1),
        f"{prefix}_simpson": 1 - (p ** 2).sum(axis=1),
        f"{prefix}_richness": (x > 0).sum(axis=1),
        f"{prefix}_total_abundance": row_sums,
    })


def aggregate_pattern_features(matrix: pd.DataFrame, prefix: str, patterns: dict[str, str]) -> pd.DataFrame:
    numeric_cols = [c for c in matrix.select_dtypes(include="number").columns if c != ID_COL]
    out = pd.DataFrame({ID_COL: matrix[ID_COL].astype(str)})
    for name, pattern in patterns.items():
        matches = [c for c in numeric_cols if re.search(pattern, c, re.I)]
        summed = matrix[matches].sum(axis=1) if matches else pd.Series(0.0, index=matrix.index)
        out[f"{prefix}_{name}"] = summed.to_numpy()
        out[f"{prefix}_{name}_present"] = (summed > 0).astype(float).to_numpy()
        out[f"{prefix}_{name}_matched_feature_count"] = len(matches)
        print(f"{prefix} {name}: matched {len(matches)} features")
    return out


def nitrogen_pathway_features(matrix: pd.DataFrame, prefix: str) -> pd.DataFrame | None:
    numeric_cols = [c for c in matrix.select_dtypes(include="number").columns if c != ID_COL]
    nitrogen_cols = [c for c in numeric_cols if NITROGEN_PATHWAY_RE.search(c)]
    print(f"{prefix}: matched {len(nitrogen_cols)} nitrogen pathway columns")
    if not nitrogen_cols:
        return None
    out = matrix[[ID_COL] + nitrogen_cols].copy()
    out[f"{prefix}_nitrogen_pathway_total"] = out[nitrogen_cols].sum(axis=1)
    out[f"{prefix}_nitrogen_pathway_present_any"] = (out[nitrogen_cols].sum(axis=1) > 0).astype(float)
    out[f"{prefix}_nitrogen_pathway_matched_feature_count"] = len(nitrogen_cols)
    return out


def build_oral_features() -> pd.DataFrame:
    if CACHE_PATHS["oral_features"].exists() and not FORCE_REBUILD_ORAL_FEATURES:
        oral = pd.read_pickle(CACHE_PATHS["oral_features"])
        print("Loaded cached oral features", oral.shape)
        return oral

    primary, pl = load_pheno_main_table("oral_microbiome", "oral_microbiome", return_loader=True)
    dataset_dir = phenoloader_dataset_dir(pl)
    parts = [primary[[ID_COL]].copy()]
    loaded = []

    for prefix, path_regex, kind in ORAL_BULK_SPECS:
        bulk, path = load_optional_bulk_table(primary, path_regex, dataset_dir)
        if bulk is None:
            continue
        wide = normalize_wide_microbiome_table(bulk, prefix)
        if wide is None:
            print("Could not normalize", prefix)
            continue
        loaded.append({"prefix": prefix, "kind": kind, "path": str(path), "participants": wide.shape[0], "features": wide.shape[1] - 1})
        if kind == "taxa":
            parts.append(alpha_diversity_from_matrix(wide, prefix))
            parts.append(aggregate_pattern_features(wide, prefix, RELEVANT_TAXA_PATTERNS))
        elif kind == "pathway":
            nitrogen = nitrogen_pathway_features(wide, prefix)
            if nitrogen is not None:
                parts.append(nitrogen)

    oral = parts[0]
    for part in parts[1:]:
        oral = oral.merge(part, on=ID_COL, how="outer")
    oral = normalize_ids(oral)

    # Mechanistic nitrate balance scores.
    pos_patterns = ["neisseria", "rothia"]
    neg_patterns = ["prevotella", "veillonella", "megasphaera"]
    num_cols = oral.select_dtypes(include="number").columns.tolist()
    pos_cols = [c for c in num_cols if any(p in c for p in pos_patterns) and not c.endswith("present") and not c.endswith("matched_feature_count")]
    neg_cols = [c for c in num_cols if any(p in c for p in neg_patterns) and not c.endswith("present") and not c.endswith("matched_feature_count")]
    oral["oral_nitrate_balance_log_ratio"] = np.log1p(oral[pos_cols].sum(axis=1)) - np.log1p(oral[neg_cols].sum(axis=1)) if pos_cols and neg_cols else np.nan
    oral["oral_nitrate_positive_score"] = oral[pos_cols].sum(axis=1) if pos_cols else np.nan
    oral["oral_nitrate_anaerobe_score"] = oral[neg_cols].sum(axis=1) if neg_cols else np.nan

    print("Loaded oral tables")
    display(pd.DataFrame(loaded))
    oral.to_pickle(CACHE_PATHS["oral_features"])
    return oral


oral_features = build_oral_features()
print("oral_features", oral_features.shape)
pd.Series(oral_features.columns).head(120)


In [ ]:
def select_relevant_outcomes(oral: pd.DataFrame) -> list[str]:
    numeric_cols = [c for c in oral.select_dtypes(include="number").columns if c != ID_COL]
    outcomes = []
    priority = [
        "oral_nitrate_balance_log_ratio", "oral_nitrate_positive_score", "oral_nitrate_anaerobe_score",
        "shannon", "simpson", "richness",
        "neisseria", "rothia", "rothia_dentocariosa", "prevotella", "veillonella", "megasphaera", "moraxellaceae", "burkholderiaceae",
        "nitrogen_pathway_total", "nitrogen_pathway_present_any",
    ]
    for pat in priority:
        for col in numeric_cols:
            if col not in outcomes and pat in col and not col.endswith("matched_feature_count"):
                outcomes.append(col)
    return outcomes


ORAL_OUTCOME_COLS = select_relevant_outcomes(oral_features)
print("Selected relevant oral outcomes", len(ORAL_OUTCOME_COLS))
ORAL_OUTCOME_COLS[:120]


## Confounders: Age, Sex, BMI, Smoking, Alcohol, And Logging Intensity


In [ ]:
def load_pheno_table(dataset: str, table: str) -> pd.DataFrame | None:
    try:
        pl = make_phenoloader(dataset)
        if table in getattr(pl, "dfs", {}):
            return normalize_ids(flatten_index(pl.dfs[table]))
        return normalize_ids(flatten_index(pl[table]))
    except Exception as exc:
        print(f"Could not load {dataset}/{table}: {exc}")
        return None


def resolve_possible_columns(df: pd.DataFrame, possible_cols: list[str]) -> list[str]:
    exact = [c for c in possible_cols if c in df.columns]
    if exact:
        return exact
    lower_to_col = {str(c).lower(): c for c in df.columns}
    found = []
    for wanted in possible_cols:
        wanted_l = wanted.lower()
        for col_l, col in lower_to_col.items():
            if wanted_l == col_l or wanted_l in col_l:
                found.append(col)
                break
    return list(dict.fromkeys(found))


def collapse_participant_table(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    keep = [ID_COL] + [c for c in cols if c in df.columns]
    tmp = df[keep].copy()
    numeric = tmp.select_dtypes(include="number").columns.difference([ID_COL]).tolist()
    other = [c for c in tmp.columns if c not in [ID_COL] + numeric]
    parts = []
    if numeric:
        parts.append(tmp.groupby(ID_COL, as_index=False)[numeric].mean())
    for c in other:
        parts.append(tmp.groupby(ID_COL)[c].agg(lambda s: s.dropna().iloc[-1] if len(s.dropna()) else np.nan).reset_index())
    if not parts:
        return pd.DataFrame(columns=[ID_COL])
    out = parts[0]
    for part in parts[1:]:
        out = out.merge(part, on=ID_COL, how="outer")
    return out


def build_confounder_table() -> tuple[pd.DataFrame, pd.DataFrame]:
    if CACHE_PATHS["confounders"].exists() and not FORCE_REBUILD_CONFOUNDERS:
        conf = pd.read_pickle(CACHE_PATHS["confounders"])
        print("Loaded cached confounders", conf.shape)
        return conf, pd.DataFrame()

    parts, summary = [], []

    for alias, specs in {
        "age": [("oral_microbiome", "age_sex", ["age"]), ("lifestyle_and_environment", "age_sex", ["age"])],
        "sex": [("oral_microbiome", "age_sex", ["sex"]), ("lifestyle_and_environment", "age_sex", ["sex"])],
        "bmi": [("lifestyle_and_environment", "lifestyle_and_environment", ["bmi", "body_mass_index"]), ("anthropometrics", "anthropometrics", ["bmi", "body_mass_index"])],
        "smoking_current_status": [("lifestyle_and_environment", "lifestyle_and_environment", SMOKING_COLUMNS)],
        "smoking_total_above_100": [("lifestyle_and_environment", "lifestyle_and_environment", ["smoking_total_above_100"])],
        "alcohol_current_frequency": [("lifestyle_and_environment", "lifestyle_and_environment", ALCOHOL_COLUMNS)],
    }.items():
        found = False
        for dataset, table, possible in specs:
            df = load_pheno_table(dataset, table)
            if df is None:
                continue
            present = resolve_possible_columns(df, possible)
            if not present:
                continue
            part = collapse_participant_table(df, present)
            part = part.rename(columns={present[0]: alias})[[ID_COL, alias]]
            parts.append(part)
            summary.append({"covariate": alias, "dataset": dataset, "table": table, "column_used": present[0], "other_matches": present[1:], "rows": len(part)})
            found = True
            break
        if not found:
            summary.append({"covariate": alias, "dataset": None, "table": None, "column_used": None, "other_matches": [], "rows": 0})

    conf = pd.DataFrame(columns=[ID_COL])
    for part in parts:
        conf = part if conf.empty else conf.merge(part, on=ID_COL, how="outer")
    conf = normalize_ids(conf) if not conf.empty else conf
    conf.to_pickle(CACHE_PATHS["confounders"])
    return conf, pd.DataFrame(summary)


confounders, confounder_source_summary = build_confounder_table()
display(confounder_source_summary)
confounders.head()


## 4. Build Analysis Tables


In [ ]:
def standardize_sex_value(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    if s in {"f", "female", "woman", "women", "0", "0.0"} or "female" in s:
        return "female"
    if s in {"m", "male", "man", "men", "1", "1.0"} or "male" in s:
        return "male"
    return np.nan


def build_analysis_table(exposure_name: str) -> pd.DataFrame:
    exposure = exposure_tables[exposure_name].copy()
    merged = exposure.merge(oral_features[[ID_COL] + ORAL_OUTCOME_COLS], on=ID_COL, how="inner")
    if not confounders.empty:
        merged = merged.merge(confounders, on=ID_COL, how="left")
    if "sex" in merged.columns:
        merged["sex_group"] = merged["sex"].map(standardize_sex_value)
        merged["sex_female"] = merged["sex_group"].map({"male": 0.0, "female": 1.0})
    else:
        merged["sex_group"] = np.nan
        merged["sex_female"] = np.nan
    if "age" in merged.columns:
        merged["age"] = pd.to_numeric(merged["age"], errors="coerce")
        merged["age_z"] = (merged["age"] - merged["age"].mean()) / merged["age"].std(ddof=0)
    return merged


analysis_tables = {name: build_analysis_table(name) for name in exposure_tables}
for name, df in analysis_tables.items():
    print(name, df.shape, "sex", df.get("sex_group", pd.Series(dtype=object)).value_counts(dropna=False).to_dict())
    display(df[[c for c in [ID_COL, "age", "age_z", "sex_group", "sex_female", "effective_exposure_metric", "exposure_value", "absolute_exposure_g", "exposure_per_1000g_logged", "exposure_g_per_logging_day", "food_events", "logging_days", "total_logged_g", "exposure_group", "decile_group"] if c in df.columns]].head())


## 1. Nitrate/Nitrite/Nitroso Oral Ecology


In [ ]:
def design_matrix(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    present = [c for c in cols if c in df.columns]
    x = df[present].copy()
    for c in x.columns:
        if pd.api.types.is_numeric_dtype(x[c]):
            vals = pd.to_numeric(x[c], errors="coerce")
            x[c] = vals.fillna(vals.median())
        else:
            x[c] = x[c].astype("category").cat.add_categories(["missing"]).fillna("missing")
    x = pd.get_dummies(x, drop_first=True, dtype=float)
    x = x.loc[:, x.nunique(dropna=False) > 1]
    return x


def fit_ols_terms(X: pd.DataFrame, y: pd.Series, terms: list[str]) -> dict:
    X = X.copy()
    X.insert(0, "intercept", 1.0)
    X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    y = pd.to_numeric(y, errors="coerce").replace([np.inf, -np.inf], np.nan)
    ok = y.notna() & X.notna().all(axis=1)
    Xn = X.loc[ok].to_numpy(dtype=float)
    yn = y.loc[ok].to_numpy(dtype=float)
    names = X.columns.tolist()
    out = {"n": len(yn), "model_df": np.nan, "residual_df": np.nan, "r2": np.nan}
    for term in terms:
        out[f"{term}_beta"] = np.nan
        out[f"{term}_se"] = np.nan
        out[f"{term}_p"] = np.nan
        out[f"{term}_standardized_beta"] = np.nan
        out[f"{term}_standardized_se"] = np.nan
    if len(yn) <= Xn.shape[1] + 2:
        return out
    beta = np.linalg.lstsq(Xn, yn, rcond=None)[0]
    pred = Xn.dot(beta)
    resid = yn - pred
    dof = len(yn) - Xn.shape[1]
    if dof <= 0:
        return out
    sigma2 = (resid @ resid) / dof
    cov = sigma2 * np.linalg.pinv(Xn.T @ Xn)
    se = np.sqrt(np.maximum(np.diag(cov), 0))
    ss_tot = ((yn - yn.mean()) @ (yn - yn.mean()))
    out.update({"model_df": Xn.shape[1] - 1, "residual_df": dof, "r2": 1 - ((resid @ resid) / ss_tot) if ss_tot > 0 else np.nan})
    y_sd = np.nanstd(yn)
    for term in terms:
        if term not in names:
            continue
        idx = names.index(term)
        t_stat = beta[idx] / se[idx] if se[idx] > 0 else np.nan
        pval = 2 * stats.t.sf(abs(t_stat), dof) if np.isfinite(t_stat) else np.nan
        term_sd = np.nanstd(X.loc[ok, term].to_numpy(dtype=float))
        std_beta = beta[idx] * term_sd / y_sd if y_sd > 0 and np.isfinite(term_sd) else np.nan
        std_se = se[idx] * term_sd / y_sd if y_sd > 0 and np.isfinite(term_sd) else np.nan
        out[f"{term}_beta"] = beta[idx]
        out[f"{term}_se"] = se[idx]
        out[f"{term}_p"] = pval
        out[f"{term}_standardized_beta"] = std_beta
        out[f"{term}_standardized_se"] = std_se
    return out


def safe_mannwhitneyu(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]; y = y[np.isfinite(y)]
    if len(x) < MIN_GROUP_N or len(y) < MIN_GROUP_N:
        return np.nan
    if np.nanmax(np.concatenate([x, y])) == np.nanmin(np.concatenate([x, y])):
        return 1.0
    return stats.mannwhitneyu(x, y, alternative="two-sided").pvalue


def cohens_d_local(x, y):
    return cohens_d(x, y)


def exposure_oral_model(df, exposure_name, outcome, covariates):
    sub = df.copy()
    sub["x"] = pd.to_numeric(sub["log1p_exposure_value"], errors="coerce")
    covars = [c for c in covariates if c in sub.columns]
    X = pd.concat([sub[["x"]], design_matrix(sub, covars)], axis=1)
    fit = fit_ols_terms(X, sub[outcome], ["x"])
    low = finite_numeric(sub.loc[sub["exposure_group"] == "low", outcome])
    high = finite_numeric(sub.loc[sub["exposure_group"] == "high", outcome])
    return {
        "analysis": "adjusted_exposure_oral_ecology",
        "exposure": exposure_name,
        "outcome": outcome,
        "metric": sub["effective_exposure_metric"].iloc[0] if "effective_exposure_metric" in sub.columns and len(sub) else PRIMARY_EXPOSURE_METRIC,
        "n": fit["n"], "model_df": fit["model_df"], "residual_df": fit["residual_df"], "r2": fit["r2"],
        "exposure_beta": fit["x_beta"], "exposure_se": fit["x_se"], "exposure_p": fit["x_p"], "exposure_standardized_beta": fit["x_standardized_beta"],
        "n_low": len(low), "n_high": len(high),
        "low_mean": float(np.mean(low)) if len(low) else np.nan,
        "high_mean": float(np.mean(high)) if len(high) else np.nan,
        "high_minus_low_d": cohens_d_local(low, high),
        "high_vs_low_p": safe_mannwhitneyu(low, high),
        "median_diff_high_minus_low": float(np.median(high) - np.median(low)) if len(low) and len(high) else np.nan,
    }


def demographic_oral_model(df, exposure_name, outcome, demographic, covariates):
    sub = df.copy()
    if demographic == "age":
        term = "age_z"
        exclude = {"age", "age_z"}
    else:
        term = "sex_female"
        exclude = {"sex", "sex_group", "sex_female"}
    if term not in sub.columns:
        return None
    covars = [c for c in covariates if c in sub.columns and c not in exclude]
    X = pd.concat([sub[[term]], design_matrix(sub, covars)], axis=1)
    fit = fit_ols_terms(X, sub[outcome], [term])
    return {
        "analysis": f"{demographic}_oral_ecology_difference",
        "exposure_context": exposure_name,
        "demographic": demographic,
        "outcome": outcome,
        "n": fit["n"], "r2": fit["r2"],
        "beta": fit[f"{term}_beta"], "se": fit[f"{term}_se"], "p_value": fit[f"{term}_p"], "standardized_beta": fit[f"{term}_standardized_beta"], "standardized_se": fit[f"{term}_standardized_se"],
    }


def select_main_oral_ecology_outcomes(oral: pd.DataFrame) -> list[str]:
    numeric_cols = [c for c in oral.select_dtypes(include="number").columns if c != ID_COL]
    wanted = [
        "oral_nitrate_balance_log_ratio", "oral_nitrate_positive_score", "oral_nitrate_anaerobe_score",
        "oral_metaphlan_genus_shannon", "oral_metaphlan_species_shannon", "oral_metaphlan_family_shannon",
        "oral_metaphlan_genus_neisseria", "oral_metaphlan_species_neisseria", "oral_metaphlan_family_neisseria",
        "oral_metaphlan_genus_rothia", "oral_metaphlan_species_rothia", "oral_metaphlan_species_rothia_dentocariosa",
        "oral_metaphlan_genus_prevotella", "oral_metaphlan_species_prevotella", "oral_metaphlan_family_prevotella",
        "oral_metaphlan_genus_veillonella", "oral_metaphlan_species_veillonella", "oral_metaphlan_family_veillonella",
        "oral_metaphlan_genus_megasphaera", "oral_metaphlan_species_megasphaera",
        "oral_metaphlan_genus_moraxellaceae", "oral_metaphlan_family_moraxellaceae",
        "oral_metaphlan_genus_burkholderiaceae", "oral_metaphlan_family_burkholderiaceae",
    ]
    out = []
    for w in wanted:
        matches = [c for c in numeric_cols if c == w or (w in c and not c.endswith("present") and not c.endswith("matched_feature_count"))]
        for c in matches:
            if c not in out:
                out.append(c)
    return out


MAIN_EXPOSURES = [x for x in ["overall_nitrate", "vegetable_nitrate", "overall_nitrite", "processed_nitrite", "nitroso_axis", "arginine_no_axis"] if x in analysis_tables]
MAIN_ORAL_OUTCOME_COLS = select_main_oral_ecology_outcomes(oral_features)
confounder_cols = [c for c in BASE_COVARIATES + LOGGING_COVARIATES if c in next(iter(analysis_tables.values())).columns]
print("Main exposures:", MAIN_EXPOSURES)
print("Main oral ecology outcomes:", len(MAIN_ORAL_OUTCOME_COLS))
display(pd.DataFrame({"outcome": MAIN_ORAL_OUTCOME_COLS}))
print("Adjusted covariates:", confounder_cols)

ecology_rows, demo_rows = [], []
for exposure_name in MAIN_EXPOSURES:
    df = analysis_tables[exposure_name]
    for outcome in MAIN_ORAL_OUTCOME_COLS:
        if outcome not in df.columns:
            continue
        ecology_rows.append(exposure_oral_model(df, exposure_name, outcome, confounder_cols))
        for demographic in ["age", "sex"]:
            row = demographic_oral_model(df, exposure_name, outcome, demographic, confounder_cols)
            if row is not None:
                demo_rows.append(row)

ecology_results = pd.DataFrame(ecology_rows)
demographic_oral_results = pd.DataFrame(demo_rows)
# Demographic oral tests do not depend on exposure context; keep one row per demographic/outcome before FDR.
demographic_oral_results = demographic_oral_results.sort_values(["demographic", "outcome", "p_value"], na_position="last").drop_duplicates(["demographic", "outcome"]).reset_index(drop=True)
ecology_results["exposure_q_value"] = p_adjust_bh(ecology_results["exposure_p"])
ecology_results["high_vs_low_q_value"] = p_adjust_bh(ecology_results["high_vs_low_p"])
demographic_oral_results["q_value"] = p_adjust_bh(demographic_oral_results["p_value"])

print("Adjusted nitrate/nitrite/nitroso oral ecology results, small prespecified FDR family")
display(ecology_results.sort_values(["exposure_q_value", "exposure_p"], na_position="last").head(80))
print("Age and sex differences in oral ecology, small prespecified FDR family")
display(demographic_oral_results.sort_values(["q_value", "p_value"], na_position="last").head(80))


## 2. Processed Nitrite To Nitroso-Axis Pathway Analysis

This section tests whether the processed-nitrite association with oral ecology is consistent with a nitrosation-context pathway. It models processed nitrite as the upstream diet axis, the KG-derived nitroso/nitrosamine axis as a mediator-like variable, and prespecified oral ecology outcomes as endpoints.

Interpretation guardrail: this is an observational attenuation/pathway-consistency analysis, not proof that measured N-nitroso compounds causally mediate the effect.


In [ ]:
NITRITE_UPSTREAM_EXPOSURE = "processed_nitrite"
NITROSO_MEDIATOR_EXPOSURE = "nitroso_axis"


def sobel_p_value(a, a_se, b, b_se):
    if not all(np.isfinite(v) for v in [a, a_se, b, b_se]) or a_se <= 0 or b_se <= 0:
        return np.nan
    se_ab = np.sqrt((b ** 2) * (a_se ** 2) + (a ** 2) * (b_se ** 2))
    if not np.isfinite(se_ab) or se_ab <= 0:
        return np.nan
    z = (a * b) / se_ab
    return 2 * stats.norm.sf(abs(z))


def build_processed_nitrite_nitroso_table():
    missing = [x for x in [NITRITE_UPSTREAM_EXPOSURE, NITROSO_MEDIATOR_EXPOSURE] if x not in analysis_tables]
    if missing:
        raise KeyError(f"Missing required exposure tables: {missing}")

    base = analysis_tables[NITRITE_UPSTREAM_EXPOSURE].copy()
    mediator_cols = [ID_COL, "log1p_exposure_value", "exposure_value", "effective_exposure_metric"]
    mediator = analysis_tables[NITROSO_MEDIATOR_EXPOSURE][[c for c in mediator_cols if c in analysis_tables[NITROSO_MEDIATOR_EXPOSURE].columns]].copy()
    mediator = mediator.rename(columns={
        "log1p_exposure_value": "nitroso_log1p_exposure_value",
        "exposure_value": "nitroso_exposure_value",
        "effective_exposure_metric": "nitroso_effective_exposure_metric",
    })
    merged = base.merge(mediator, on=ID_COL, how="inner")
    merged["processed_nitrite_log1p_exposure_value"] = pd.to_numeric(merged["log1p_exposure_value"], errors="coerce")
    merged["nitroso_log1p_exposure_value"] = pd.to_numeric(merged["nitroso_log1p_exposure_value"], errors="coerce")
    return merged


def nitrite_nitroso_pathway_model(df, outcome, covariates):
    sub = df.copy()
    sub["processed_x"] = pd.to_numeric(sub["processed_nitrite_log1p_exposure_value"], errors="coerce")
    sub["nitroso_mediator"] = pd.to_numeric(sub["nitroso_log1p_exposure_value"], errors="coerce")
    sub["y"] = pd.to_numeric(sub[outcome], errors="coerce")

    covars = [c for c in covariates if c in sub.columns]
    covar_X = design_matrix(sub, covars)

    a_fit = fit_ols_terms(pd.concat([sub[["processed_x"]], covar_X], axis=1), sub["nitroso_mediator"], ["processed_x"])
    total_fit = fit_ols_terms(pd.concat([sub[["processed_x"]], covar_X], axis=1), sub["y"], ["processed_x"])
    direct_fit = fit_ols_terms(pd.concat([sub[["processed_x", "nitroso_mediator"]], covar_X], axis=1), sub["y"], ["processed_x", "nitroso_mediator"])

    a = a_fit["processed_x_beta"]
    a_se = a_fit["processed_x_se"]
    b = direct_fit["nitroso_mediator_beta"]
    b_se = direct_fit["nitroso_mediator_se"]
    indirect = a * b if np.isfinite(a) and np.isfinite(b) else np.nan
    total = total_fit["processed_x_beta"]
    direct = direct_fit["processed_x_beta"]
    attenuation = (total - direct) / total if np.isfinite(total) and abs(total) > 1e-12 and np.isfinite(direct) else np.nan

    return {
        "analysis": "processed_nitrite_to_nitroso_axis_pathway",
        "upstream_exposure": NITRITE_UPSTREAM_EXPOSURE,
        "mediator_exposure": NITROSO_MEDIATOR_EXPOSURE,
        "outcome": outcome,
        "n": direct_fit["n"],
        "processed_metric": sub["effective_exposure_metric"].iloc[0] if "effective_exposure_metric" in sub.columns and len(sub) else PRIMARY_EXPOSURE_METRIC,
        "nitroso_metric": sub["nitroso_effective_exposure_metric"].iloc[0] if "nitroso_effective_exposure_metric" in sub.columns and len(sub) else PRIMARY_EXPOSURE_METRIC,
        "a_processed_to_nitroso_beta": a,
        "a_processed_to_nitroso_se": a_se,
        "a_processed_to_nitroso_p": a_fit["processed_x_p"],
        "a_processed_to_nitroso_standardized_beta": a_fit["processed_x_standardized_beta"],
        "total_processed_to_outcome_beta": total,
        "total_processed_to_outcome_p": total_fit["processed_x_p"],
        "total_processed_to_outcome_standardized_beta": total_fit["processed_x_standardized_beta"],
        "direct_processed_to_outcome_beta_after_nitroso": direct,
        "direct_processed_to_outcome_p_after_nitroso": direct_fit["processed_x_p"],
        "direct_processed_to_outcome_standardized_beta_after_nitroso": direct_fit["processed_x_standardized_beta"],
        "b_nitroso_to_outcome_beta_adjusted": b,
        "b_nitroso_to_outcome_se_adjusted": b_se,
        "b_nitroso_to_outcome_p_adjusted": direct_fit["nitroso_mediator_p"],
        "b_nitroso_to_outcome_standardized_beta_adjusted": direct_fit["nitroso_mediator_standardized_beta"],
        "indirect_beta_product_ab": indirect,
        "indirect_sobel_p": sobel_p_value(a, a_se, b, b_se),
        "fraction_attenuated_after_nitroso": attenuation,
        "total_model_r2": total_fit["r2"],
        "direct_model_r2": direct_fit["r2"],
    }


def pathway_compact_label(label):
    label = str(label)
    replacements = {
        "oral_metaphlan_": "",
        "nitrate_reducer_": "",
        "pathway_abundance_pathway_level_": "",
        "pathway_coverage_pathway_level_": "",
        "k__bacteria_p__": "",
        "c__": "",
        "o__": "",
        "f__": "",
        "g__": "",
        "s__": "",
        "_": " ",
    }
    for old, new in replacements.items():
        label = label.replace(old, new)
    return " ".join(label.split())


def plot_nitrite_nitroso_pathway(results: pd.DataFrame, top_n: int = 20):
    if results.empty or "indirect_beta_product_ab" not in results.columns:
        return None, None
    sub = results.copy()
    sub = sub[np.isfinite(pd.to_numeric(sub["indirect_beta_product_ab"], errors="coerce"))].copy()
    if sub.empty:
        return None, None
    sub["abs_indirect"] = sub["indirect_beta_product_ab"].abs()
    sub = sub.sort_values(["indirect_q_value", "indirect_sobel_p", "abs_indirect"], ascending=[True, True, False], na_position="last").head(top_n)
    sub = sub.iloc[::-1]

    fig, ax = plt.subplots(figsize=(10, max(4, 0.34 * len(sub))), dpi=140)
    colors = np.where(sub["indirect_beta_product_ab"] >= 0, "#b2182b", "#2166ac")
    ax.barh(np.arange(len(sub)), sub["indirect_beta_product_ab"], color=colors, alpha=0.84)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_yticks(np.arange(len(sub)))
    ax.set_yticklabels([pathway_compact_label(x)[-86:] for x in sub["outcome"]], fontsize=8)
    ax.set_xlabel("Indirect product a*b: processed nitrite -> nitroso axis -> oral outcome")
    ax.set_title("Processed nitrite to nitroso-axis pathway signal")
    ax.grid(True, axis="x", color="black", alpha=0.18, linewidth=0.8)
    ax.set_facecolor("white")
    fig.patch.set_facecolor("white")
    for i, (_, row) in enumerate(sub.iterrows()):
        label = f"q={row['indirect_q_value']:.3g}" if np.isfinite(row.get("indirect_q_value", np.nan)) else "q=NA"
        x = row["indirect_beta_product_ab"]
        ax.text(x, i, " " + label if x >= 0 else label + " ", va="center", ha="left" if x >= 0 else "right", fontsize=7)
    plt.tight_layout()
    return fig, ax


nitrite_nitroso_pathway_table = build_processed_nitrite_nitroso_table()
print("Processed nitrite/nitroso pathway table:", nitrite_nitroso_pathway_table.shape)
print("Upstream metric:", nitrite_nitroso_pathway_table["effective_exposure_metric"].iloc[0] if "effective_exposure_metric" in nitrite_nitroso_pathway_table.columns and len(nitrite_nitroso_pathway_table) else PRIMARY_EXPOSURE_METRIC)
print("Mediator metric:", nitrite_nitroso_pathway_table["nitroso_effective_exposure_metric"].iloc[0] if "nitroso_effective_exposure_metric" in nitrite_nitroso_pathway_table.columns and len(nitrite_nitroso_pathway_table) else PRIMARY_EXPOSURE_METRIC)

nitrite_nitroso_rows = []
for outcome in MAIN_ORAL_OUTCOME_COLS:
    nitrite_nitroso_rows.append(nitrite_nitroso_pathway_model(nitrite_nitroso_pathway_table, outcome, confounder_cols))

nitrite_nitroso_results = pd.DataFrame(nitrite_nitroso_rows)
if not nitrite_nitroso_results.empty:
    nitrite_nitroso_results["a_q_value"] = p_adjust_bh(nitrite_nitroso_results["a_processed_to_nitroso_p"])
    nitrite_nitroso_results["total_q_value"] = p_adjust_bh(nitrite_nitroso_results["total_processed_to_outcome_p"])
    nitrite_nitroso_results["direct_q_value"] = p_adjust_bh(nitrite_nitroso_results["direct_processed_to_outcome_p_after_nitroso"])
    nitrite_nitroso_results["b_q_value"] = p_adjust_bh(nitrite_nitroso_results["b_nitroso_to_outcome_p_adjusted"])
    nitrite_nitroso_results["indirect_q_value"] = p_adjust_bh(nitrite_nitroso_results["indirect_sobel_p"])
    nitrite_nitroso_results["abs_indirect_beta"] = nitrite_nitroso_results["indirect_beta_product_ab"].abs()

print("Processed nitrite -> nitroso axis -> oral ecology results")
display(nitrite_nitroso_results.sort_values(["indirect_q_value", "indirect_sobel_p", "abs_indirect_beta"], ascending=[True, True, False], na_position="last").head(80))

nitrite_nitroso_summary = pd.DataFrame([{
    "tested_outcomes": int(nitrite_nitroso_results["indirect_sobel_p"].notna().sum()) if not nitrite_nitroso_results.empty else 0,
    "indirect_fdr_lt_0_05": int((nitrite_nitroso_results["indirect_q_value"] < 0.05).sum()) if not nitrite_nitroso_results.empty else 0,
    "indirect_fdr_lt_0_10": int((nitrite_nitroso_results["indirect_q_value"] < 0.10).sum()) if not nitrite_nitroso_results.empty else 0,
    "min_indirect_q": float(nitrite_nitroso_results["indirect_q_value"].min()) if not nitrite_nitroso_results.empty else np.nan,
    "median_abs_fraction_attenuated": float(nitrite_nitroso_results["fraction_attenuated_after_nitroso"].abs().median()) if not nitrite_nitroso_results.empty else np.nan,
}])
display(nitrite_nitroso_summary)

fig, ax = plot_nitrite_nitroso_pathway(nitrite_nitroso_results)
plt.show()
# fig.savefig(OUTPUT_DIR / "processed_nitrite_to_nitroso_axis_pathway.png", dpi=300, bbox_inches="tight")


## 3. Age And Sex Mediation


In [ ]:
def mediation_row(df: pd.DataFrame, exposure_name: str, outcome: str, upstream_label: str, upstream_col: str, exclude_cols: set[str], covariates: list[str]) -> dict:
    sub = df.copy()
    sub["mediator_x"] = pd.to_numeric(sub["log1p_exposure_value"], errors="coerce")
    sub["upstream_u"] = pd.to_numeric(sub[upstream_col], errors="coerce")
    covars = [c for c in covariates if c in sub.columns and c not in exclude_cols]
    covar_X = design_matrix(sub, covars)

    a_fit = fit_ols_terms(pd.concat([sub[["upstream_u"]], covar_X], axis=1), sub["mediator_x"], ["upstream_u"])
    total_fit = fit_ols_terms(pd.concat([sub[["upstream_u"]], covar_X], axis=1), sub[outcome], ["upstream_u"])
    direct_fit = fit_ols_terms(pd.concat([sub[["upstream_u", "mediator_x"]], covar_X], axis=1), sub[outcome], ["upstream_u", "mediator_x"])

    a, a_se = a_fit.get("upstream_u_beta", np.nan), a_fit.get("upstream_u_se", np.nan)
    b, b_se = direct_fit.get("mediator_x_beta", np.nan), direct_fit.get("mediator_x_se", np.nan)
    indirect = a * b if np.isfinite(a) and np.isfinite(b) else np.nan
    sobel_se = math.sqrt((b ** 2) * (a_se ** 2) + (a ** 2) * (b_se ** 2)) if np.isfinite(a) and np.isfinite(b) and np.isfinite(a_se) and np.isfinite(b_se) else np.nan
    sobel_z = indirect / sobel_se if np.isfinite(indirect) and np.isfinite(sobel_se) and sobel_se > 0 else np.nan
    indirect_p = 2 * stats.norm.sf(abs(sobel_z)) if np.isfinite(sobel_z) else np.nan
    total_beta = total_fit.get("upstream_u_beta", np.nan)
    direct_beta = direct_fit.get("upstream_u_beta", np.nan)
    attenuation = (total_beta - direct_beta) / total_beta if np.isfinite(total_beta) and total_beta != 0 and np.isfinite(direct_beta) else np.nan
    return {
        "analysis": f"{upstream_label}_diet_mediation_style",
        "upstream": upstream_label,
        "exposure": exposure_name,
        "outcome": outcome,
        "n": direct_fit.get("n", np.nan),
        "a_upstream_to_diet_beta": a,
        "a_upstream_to_diet_p": a_fit.get("upstream_u_p", np.nan),
        "b_diet_to_outcome_beta_adjusted": b,
        "b_diet_to_outcome_p_adjusted": direct_fit.get("mediator_x_p", np.nan),
        "total_upstream_to_outcome_beta": total_beta,
        "total_upstream_to_outcome_p": total_fit.get("upstream_u_p", np.nan),
        "direct_upstream_to_outcome_beta_after_diet": direct_beta,
        "direct_upstream_to_outcome_p_after_diet": direct_fit.get("upstream_u_p", np.nan),
        "indirect_beta_product_ab": indirect,
        "indirect_sobel_p": indirect_p,
        "fraction_attenuated_after_diet": attenuation,
        "mediator_model_r2": a_fit.get("r2", np.nan),
        "outcome_total_model_r2": total_fit.get("r2", np.nan),
        "outcome_direct_model_r2": direct_fit.get("r2", np.nan),
    }


mediation_specs = [
    ("age", "age_z", {"age", "age_z"}),
    ("sex_female", "sex_female", {"sex", "sex_group", "sex_female"}),
]
mediation_rows = []
for exposure_name in MAIN_EXPOSURES:
    df = analysis_tables[exposure_name]
    for upstream_label, upstream_col, exclude_cols in mediation_specs:
        if upstream_col not in df.columns:
            continue
        for outcome in MAIN_ORAL_OUTCOME_COLS:
            mediation_rows.append(mediation_row(df, exposure_name, outcome, upstream_label, upstream_col, exclude_cols, confounder_cols))

mediation_results = pd.DataFrame(mediation_rows)
if not mediation_results.empty:
    mediation_results["a_q_value"] = p_adjust_bh(mediation_results["a_upstream_to_diet_p"])
    mediation_results["b_q_value"] = p_adjust_bh(mediation_results["b_diet_to_outcome_p_adjusted"])
    mediation_results["total_upstream_q_value"] = p_adjust_bh(mediation_results["total_upstream_to_outcome_p"])
    mediation_results["direct_upstream_q_value"] = p_adjust_bh(mediation_results["direct_upstream_to_outcome_p_after_diet"])
    mediation_results["indirect_q_value"] = p_adjust_bh(mediation_results["indirect_sobel_p"])
    mediation_results["abs_indirect_beta"] = mediation_results["indirect_beta_product_ab"].abs()

print("Age and sex mediation-style results. Observational Sobel/attenuation screen; interpret as pathway consistency, not causal proof.")
display(mediation_results.sort_values(["indirect_q_value", "indirect_sobel_p", "abs_indirect_beta"], ascending=[True, True, False], na_position="last").head(100))

print("Mediation summary by upstream variable")
mediation_overview = mediation_results.groupby("upstream", as_index=False).agg(
    tests=("indirect_sobel_p", "size"),
    indirect_fdr_lt_005=("indirect_q_value", lambda s: int((s < 0.05).sum())),
    indirect_fdr_lt_010=("indirect_q_value", lambda s: int((s < 0.10).sum())),
    min_indirect_q=("indirect_q_value", "min"),
    median_abs_attenuation=("fraction_attenuated_after_diet", lambda s: float(np.nanmedian(np.abs(s)))),
)
display(mediation_overview)
if not mediation_overview.empty:
    sig = mediation_overview[mediation_overview["indirect_fdr_lt_005"] > 0]
    if len(sig):
        print("Mediation screen summary: some indirect paths survive FDR q<0.05. Treat as observational/sensitivity evidence and inspect effect sizes/attenuation, not causal mediation proof.")
    else:
        print("Mediation screen summary: no indirect paths survive FDR q<0.05; this argues against strong diet-mediated demographic effects in this specification.")


## 4. Age And Sex Diet Differences Correlated With Oral Ecology


In [ ]:
def setup_axis(ax, title, x_title, y_title):
    ax.set_title(title, fontsize=13, color="black", pad=12)
    ax.set_xlabel(x_title, fontsize=12, color="black")
    ax.set_ylabel(y_title, fontsize=12, color="black")
    ax.set_facecolor("white")
    ax.figure.patch.set_facecolor("white")
    ax.grid(True, color="black", alpha=0.25, linewidth=0.7)
    for spine in ax.spines.values():
        spine.set_color("black")
    ax.tick_params(axis="both", colors="black", labelsize=11)


def compact_label(value):
    text = pretty_label(value)
    text = text.replace("oral metaphlan genus ", "genus ")
    text = text.replace("oral metaphlan species ", "species ")
    text = text.replace("oral metaphlan family ", "family ")
    return text


def plot_ecology_effect_heatmap(results, title="Adjusted diet-to-oral ecology beta; cells show FDR q"):
    sub = results.copy()
    sub = sub[sub["outcome"].isin(MAIN_ORAL_OUTCOME_COLS) & sub["exposure"].isin(MAIN_EXPOSURES)]
    mat = sub.pivot_table(index="outcome", columns="exposure", values="exposure_standardized_beta", aggfunc="first").reindex(index=MAIN_ORAL_OUTCOME_COLS, columns=MAIN_EXPOSURES)
    qmat = sub.pivot_table(index="outcome", columns="exposure", values="exposure_q_value", aggfunc="first").reindex(index=MAIN_ORAL_OUTCOME_COLS, columns=MAIN_EXPOSURES)
    vmax = np.nanmax(np.abs(mat.to_numpy(dtype=float))) if mat.size else 1
    vmax = vmax if np.isfinite(vmax) and vmax > 0 else 1
    fig, ax = plt.subplots(figsize=(11, max(5, 0.42 * len(MAIN_ORAL_OUTCOME_COLS))), dpi=140)
    im = ax.imshow(mat.to_numpy(dtype=float), cmap="RdBu_r", aspect="auto", vmin=-vmax, vmax=vmax)
    ax.set_xticks(np.arange(len(MAIN_EXPOSURES)))
    ax.set_xticklabels([pretty_label(x) for x in MAIN_EXPOSURES], rotation=35, ha="right")
    ax.set_yticks(np.arange(len(MAIN_ORAL_OUTCOME_COLS)))
    ax.set_yticklabels([compact_label(x)[-88:] for x in MAIN_ORAL_OUTCOME_COLS], fontsize=8)
    ax.set_title(title, fontsize=13)
    for i in range(len(MAIN_ORAL_OUTCOME_COLS)):
        for j in range(len(MAIN_EXPOSURES)):
            q = qmat.iloc[i, j]
            if np.isfinite(q):
                ax.text(j, i, f"q={q:.2g}", ha="center", va="center", fontsize=7, color="black")
    cb = fig.colorbar(im, ax=ax)
    cb.set_label("Standardized beta")
    ax.figure.patch.set_facecolor("white")
    fig.tight_layout()
    return fig, ax


def plot_demographic_oral_effects(results, demographic):
    sub = results[results["demographic"] == demographic].copy()
    sub = sub.sort_values(["q_value", "p_value"], na_position="last").head(22).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9.4, max(4.5, 0.38 * len(sub) + 1.5)), dpi=140)
    y = np.arange(len(sub))
    vals = pd.to_numeric(sub["standardized_beta"], errors="coerce")
    se_col = "standardized_se" if "standardized_se" in sub.columns else "se"
    xerr = 1.96 * pd.to_numeric(sub[se_col], errors="coerce").fillna(0)
    colors = ["#b2182b" if v > 0 else "#2166ac" for v in vals]
    ax.errorbar(vals, y, xerr=xerr, fmt="none", ecolor="black", capsize=4)
    ax.scatter(vals, y, color=colors, edgecolor="black", zorder=3)
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels([compact_label(x)[-82:] for x in sub["outcome"]], fontsize=8)
    for yi, (_, row) in enumerate(sub.iterrows()):
        ax.text(row["standardized_beta"], yi + 0.16, f"p={row['p_value']:.2g}; q={row['q_value']:.2g}", ha="center", fontsize=7)
    setup_axis(ax, f"{demographic} differences in oral ecology", "Standardized beta", "Oral outcome")
    fig.tight_layout()
    return fig, ax


def plot_group_violin(df, exposure_name, outcome):
    order = GROUP_ORDER
    vals = [pd.to_numeric(df.loc[df["exposure_group"] == g, outcome], errors="coerce").dropna().to_numpy(float) for g in order]
    row = ecology_results[(ecology_results["exposure"] == exposure_name) & (ecology_results["outcome"] == outcome)]
    row = row.iloc[0].to_dict() if len(row) else {}
    fig, ax = plt.subplots(figsize=(7.6, 5.0), dpi=140)
    pos = np.arange(1, len(order)+1)
    safe_vals = [v if len(v) else np.array([np.nan]) for v in vals]
    parts = ax.violinplot(safe_vals, positions=pos, widths=0.72, showmeans=False, showextrema=False, showmedians=False)
    for body, group in zip(parts["bodies"], order):
        body.set_facecolor(GROUP_COLORS[group]); body.set_edgecolor("black"); body.set_alpha(0.45)
    box = ax.boxplot(safe_vals, positions=pos, patch_artist=True, widths=0.32, showfliers=False)
    for patch, group in zip(box["boxes"], order):
        patch.set_facecolor(GROUP_COLORS[group]); patch.set_edgecolor("black"); patch.set_alpha(0.8)
    for element in ["whiskers", "caps", "medians"]:
        for item in box[element]: item.set_color("black")
    ax.set_xticks(pos)
    ax.set_xticklabels([f"{g}\nn={len(v)}" for g, v in zip(order, vals)])
    setup_axis(ax, f"{pretty_label(exposure_name)} vs {compact_label(outcome)}", "Exposure tertile", compact_label(outcome))
    ax.text(0.5, 0.98, f"adjusted p={row.get('exposure_p', np.nan):.2g}; q={row.get('exposure_q_value', np.nan):.2g}\nhigh-low p={row.get('high_vs_low_p', np.nan):.2g}; q={row.get('high_vs_low_q_value', np.nan):.2g}", transform=ax.transAxes, ha="center", va="top", fontsize=9, bbox=dict(facecolor="white", edgecolor="black", boxstyle="square,pad=0.25"))
    fig.tight_layout()
    return fig, ax


# Diet-demographic correlations and overlap with oral ecology results.
diet_demo_rows = []
for exposure_name in MAIN_EXPOSURES:
    df = analysis_tables[exposure_name].copy()
    df["x"] = pd.to_numeric(df["log1p_exposure_value"], errors="coerce")
    for demographic, term, exclude in [("age", "age_z", {"age", "age_z"}), ("sex", "sex_female", {"sex", "sex_group", "sex_female"})]:
        if term not in df.columns:
            continue
        covars = [c for c in confounder_cols if c in df.columns and c not in exclude]
        fit = fit_ols_terms(pd.concat([df[[term]], design_matrix(df, covars)], axis=1), df["x"], [term])
        diet_demo_rows.append({"demographic": demographic, "exposure": exposure_name, "n": fit["n"], "beta": fit[f"{term}_beta"], "se": fit[f"{term}_se"], "p_value": fit[f"{term}_p"], "standardized_beta": fit[f"{term}_standardized_beta"], "r2": fit["r2"]})

diet_demographic_results = pd.DataFrame(diet_demo_rows)
diet_demographic_results["q_value"] = p_adjust_bh(diet_demographic_results["p_value"])
print("Age and sex differences in diet exposure axes")
display(diet_demographic_results.sort_values(["q_value", "p_value"], na_position="last"))

fig, ax = plot_ecology_effect_heatmap(ecology_results)
plt.show()
# fig.savefig(OUTPUT_DIR / "fig_main_oral_ecology_heatmap.png", dpi=300, bbox_inches="tight")

print("Skipping older global age/sex forest plots here; use Section 5 diet-stratified age/sex violin plots for manuscript figures.")
# for demographic in ["age", "sex"]:
#     fig, ax = plot_demographic_oral_effects(demographic_oral_results, demographic)
#     plt.show()
#     # fig.savefig(OUTPUT_DIR / f"fig_{demographic}_oral_ecology_effects.png", dpi=300, bbox_inches="tight")

for _, row in ecology_results.sort_values(["exposure_q_value", "exposure_p"], na_position="last").head(12).iterrows():
    fig, ax = plot_group_violin(analysis_tables[row["exposure"]], row["exposure"], row["outcome"])
    plt.show()
    # fig.savefig(OUTPUT_DIR / f"fig_violin_{row['exposure']}_{sanitize_feature_name(row['outcome'])}.png", dpi=300, bbox_inches="tight")

# Combined evidence table: demographic -> diet and diet -> oral ecology.
demo_diet = diet_demographic_results.rename(columns={"beta": "demo_to_diet_beta", "q_value": "demo_to_diet_q"})[["demographic", "exposure", "demo_to_diet_beta", "demo_to_diet_q"]]
diet_oral = ecology_results.rename(columns={"exposure_beta": "diet_to_oral_beta", "exposure_q_value": "diet_to_oral_q"})[["exposure", "outcome", "diet_to_oral_beta", "diet_to_oral_q", "high_minus_low_d"]]
demographic_diet_oral_alignment = demo_diet.merge(diet_oral, on="exposure", how="inner")
demographic_diet_oral_alignment["same_direction_score"] = np.sign(demographic_diet_oral_alignment["demo_to_diet_beta"]) * np.sign(demographic_diet_oral_alignment["diet_to_oral_beta"])
print("Exploratory alignment only: demo_to_diet_q = age/sex difference in diet exposure; diet_to_oral_q = diet-to-oral association. This is not a within-diet-group demographic test and not mediation.")
display(demographic_diet_oral_alignment.sort_values(["demo_to_diet_q", "diet_to_oral_q"], na_position="last").head(80))


## 5. Diet-Stratified Age And Sex Oral Ecology Differences

This section answers the direct reviewer/manuscript question: within people in the low, mid, or high group for a given dietary nitrogen axis, is oral ecology different by age or sex? These are difference/stratified association plots, separate from the mediation-style analysis above.


In [ ]:
STRATIFIED_DEMO_MIN_N = 80
STRATIFIED_DEMO_TOP_N = 8


def fit_demographic_within_diet_group(df: pd.DataFrame, exposure_name: str, outcome: str, demographic: str, exposure_group: str, covariates: list[str]) -> dict:
    sub = df[df["exposure_group"] == exposure_group].copy()
    if demographic == "age":
        term = "age_z"
        exclude = {"age", "age_z"}
        analysis = "age_difference_within_diet_group"
    else:
        term = "sex_female"
        exclude = {"sex", "sex_group", "sex_female"}
        analysis = "sex_difference_within_diet_group"
    if term not in sub.columns or outcome not in sub.columns:
        return {"analysis": analysis, "exposure": exposure_name, "exposure_group": exposure_group, "demographic": demographic, "outcome": outcome, "n": 0}
    covars = [c for c in covariates if c in sub.columns and c not in exclude]
    X = pd.concat([sub[[term]], design_matrix(sub, covars)], axis=1)
    fit = fit_ols_terms(X, sub[outcome], [term])
    return {
        "analysis": analysis,
        "exposure": exposure_name,
        "exposure_group": exposure_group,
        "demographic": demographic,
        "outcome": outcome,
        "n": fit.get("n", np.nan),
        "beta": fit.get(f"{term}_beta", np.nan),
        "se": fit.get(f"{term}_se", np.nan),
        "p_value": fit.get(f"{term}_p", np.nan),
        "standardized_beta": fit.get(f"{term}_standardized_beta", np.nan),
        "r2": fit.get("r2", np.nan),
        "covariate_exclusion": "age excluded from covariates" if demographic == "age" else "sex excluded from covariates",
        "covariates_used": ", ".join(covars),
    }


stratified_demo_rows = []
for exposure_name in MAIN_EXPOSURES:
    df = analysis_tables[exposure_name]
    for exposure_group in GROUP_ORDER:
        if exposure_group not in set(df["exposure_group"].dropna()):
            continue
        for demographic in ["age", "sex"]:
            for outcome in MAIN_ORAL_OUTCOME_COLS:
                row = fit_demographic_within_diet_group(df, exposure_name, outcome, demographic, exposure_group, confounder_cols)
                stratified_demo_rows.append(row)

diet_stratified_demographic_results = pd.DataFrame(stratified_demo_rows)
if not diet_stratified_demographic_results.empty:
    diet_stratified_demographic_results.loc[diet_stratified_demographic_results["n"] < STRATIFIED_DEMO_MIN_N, "p_value"] = np.nan
    diet_stratified_demographic_results["q_value"] = p_adjust_bh(diet_stratified_demographic_results["p_value"])
    diet_stratified_demographic_results["abs_standardized_beta"] = diet_stratified_demographic_results["standardized_beta"].abs()

print("Diet-stratified demographic tests: within low/mid/high exposure group, test age or sex difference in oral ecology")
print("For age rows, age is the tested term and is excluded from covariates. For sex rows, sex is the tested term and is excluded from covariates.")
display(diet_stratified_demographic_results.sort_values(["q_value", "p_value", "abs_standardized_beta"], ascending=[True, True, False], na_position="last").head(120))


def stratified_demo_plot_row(results: pd.DataFrame, exposure_name: str, demographic: str, outcome: str, exposure_group: str):
    row = results[(results["exposure"] == exposure_name) & (results["demographic"] == demographic) & (results["outcome"] == outcome) & (results["exposure_group"] == exposure_group)]
    return row.iloc[0].to_dict() if len(row) else {}


def plot_demographic_within_diet_groups(exposure_name: str, demographic: str, outcome: str):
    df = analysis_tables[exposure_name].copy()
    fig, axes = plt.subplots(1, len(GROUP_ORDER), figsize=(14.0, 4.7), dpi=140, sharey=True)
    if len(GROUP_ORDER) == 1:
        axes = [axes]
    for ax, exposure_group in zip(axes, GROUP_ORDER):
        sub = df[df["exposure_group"] == exposure_group].copy()
        if demographic == "age":
            age_vals = pd.to_numeric(sub["age"], errors="coerce") if "age" in sub.columns else pd.Series(np.nan, index=sub.index)
            sub["age_plot_group"] = pd.qcut(age_vals.rank(method="first"), q=3, labels=["younger", "middle", "older"])
            x_col = "age_plot_group"
            order = ["younger", "middle", "older"]
            colors = {"younger": "#2166ac", "middle": "#f4a582", "older": "#b2182b"}
            x_label = "Age tertile within diet group"
        else:
            x_col = "sex_group"
            order = [g for g in ["male", "female"] if g in set(sub[x_col].dropna())]
            colors = {"male": "#2166ac", "female": "#b2182b"}
            x_label = "Sex within diet group"
        vals = [pd.to_numeric(sub.loc[sub[x_col] == g, outcome], errors="coerce").dropna().to_numpy(float) for g in order]
        safe_vals = [v if len(v) else np.array([np.nan]) for v in vals]
        pos = np.arange(1, len(order) + 1)
        if len(pos):
            parts = ax.violinplot(safe_vals, positions=pos, widths=0.72, showmeans=False, showextrema=False, showmedians=False)
            for body, group in zip(parts["bodies"], order):
                body.set_facecolor(colors.get(group, "#999999")); body.set_edgecolor("black"); body.set_alpha(0.45)
            box = ax.boxplot(safe_vals, positions=pos, patch_artist=True, widths=0.32, showfliers=False)
            for patch, group in zip(box["boxes"], order):
                patch.set_facecolor(colors.get(group, "#999999")); patch.set_edgecolor("black"); patch.set_alpha(0.8)
            for element in ["whiskers", "caps", "medians"]:
                for item in box[element]:
                    item.set_color("black")
            ax.set_xticks(pos)
            ax.set_xticklabels([f"{g}\nn={len(v)}" for g, v in zip(order, vals)], rotation=0)
        row = stratified_demo_plot_row(diet_stratified_demographic_results, exposure_name, demographic, outcome, exposure_group)
        beta = row.get("standardized_beta", np.nan)
        p = row.get("p_value", np.nan)
        q = row.get("q_value", np.nan)
        note = row.get("covariate_exclusion", "")
        ax.text(0.5, 0.98, f"beta={beta:.2g}; p={p:.2g}; q={q:.2g}\n{note}", transform=ax.transAxes, ha="center", va="top", fontsize=8, bbox=dict(facecolor="white", edgecolor="black", boxstyle="square,pad=0.2"))
        setup_axis(ax, f"{pretty_label(exposure_name)}: {exposure_group}", x_label, compact_label(outcome) if ax is axes[0] else "")
    fig.suptitle(f"{demographic} difference in {compact_label(outcome)} within diet exposure groups", y=1.03, fontsize=14)
    fig.tight_layout()
    return fig, axes


print("Selected stratified age/sex difference plots")
for demographic in ["age", "sex"]:
    examples = diet_stratified_demographic_results[diet_stratified_demographic_results["demographic"] == demographic].sort_values(["q_value", "p_value", "abs_standardized_beta"], ascending=[True, True, False], na_position="last")
    examples = examples.drop_duplicates(["exposure", "outcome"]).head(STRATIFIED_DEMO_TOP_N)
    print(f"Top {demographic} within-diet-group examples")
    display(examples[["exposure", "exposure_group", "outcome", "n", "standardized_beta", "p_value", "q_value", "covariate_exclusion"]])
    for _, row in examples.head(4).iterrows():
        fig, axes = plot_demographic_within_diet_groups(row["exposure"], demographic, row["outcome"])
        plt.show()
        # fig.savefig(OUTPUT_DIR / f"fig_{demographic}_within_{row['exposure']}_{sanitize_feature_name(row['outcome'])}.png", dpi=300, bbox_inches="tight")


def plot_mediation_indirect_effects(mediation_df: pd.DataFrame, upstream: str, top_n: int = 18):
    sub = mediation_df[mediation_df["upstream"] == upstream].copy()
    if sub.empty:
        return None, None
    sub["abs_indirect_beta"] = pd.to_numeric(sub["indirect_beta_product_ab"], errors="coerce").abs()
    sub = sub.sort_values(["indirect_q_value", "indirect_sobel_p", "abs_indirect_beta"], ascending=[True, True, False], na_position="last").head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(11, max(4.8, 0.36 * len(sub))), dpi=140)
    y = np.arange(len(sub))
    vals = pd.to_numeric(sub["indirect_beta_product_ab"], errors="coerce")
    colors = np.where(vals >= 0, "#b2182b", "#2166ac")
    labels = [f"{pretty_label(e)} | {compact_label(o)[-58:]}" for e, o in zip(sub["exposure"], sub["outcome"])]
    ax.barh(y, vals, color=colors, edgecolor="black", alpha=0.82)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    for yi, (_, row) in enumerate(sub.iterrows()):
        x = row["indirect_beta_product_ab"]
        p = row["indirect_sobel_p"]
        q = row["indirect_q_value"]
        ax.text(x, yi, f" p={p:.2g}; q={q:.2g}" if x >= 0 else f"p={p:.2g}; q={q:.2g} ", ha="left" if x >= 0 else "right", va="center", fontsize=7)
    setup_axis(ax, f"Mediation-style indirect effects: {upstream}", "Indirect product a*b", "Diet axis | oral outcome")
    fig.tight_layout()
    return fig, ax

print("Separate mediation-style plots: these are not group-difference plots")
for upstream in ["age", "sex_female"]:
    fig, ax = plot_mediation_indirect_effects(mediation_results, upstream)
    if fig is not None:
        plt.show()
        # fig.savefig(OUTPUT_DIR / f"fig_mediation_indirect_{upstream}.png", dpi=300, bbox_inches="tight")


## 6. Paper-Focused Figure Mining And Directional Taxa Lists

This section converts the model output into manuscript-friendly figures and ranked lists. It highlights age/sex oral ecology differences, taxa that increase or decrease under each dietary nitrogen axis, and taxa/features that move in opposite directions across axes.


In [ ]:
PAPER_TOP_N = 12
OPPOSITE_AXIS_TOP_N = 8
DIRECTION_LIST_TOP_N = 15


def numeric_col(df, col):
    return pd.to_numeric(df[col], errors="coerce") if col in df.columns else pd.Series(np.nan, index=df.index)


def make_directional_taxa_lists(results: pd.DataFrame, top_n: int = DIRECTION_LIST_TOP_N) -> pd.DataFrame:
    rows = []
    keep_cols = [
        "exposure", "direction", "rank", "outcome", "label", "standardized_beta",
        "adjusted_p", "q_value", "high_minus_low_d", "high_vs_low_q", "n", "r2",
    ]
    if results.empty:
        return pd.DataFrame(columns=keep_cols)
    res = results.copy()
    res["standardized_beta"] = numeric_col(res, "exposure_standardized_beta")
    res["adjusted_p"] = numeric_col(res, "exposure_p")
    res["q_value"] = numeric_col(res, "exposure_q_value")
    res["high_vs_low_q"] = numeric_col(res, "high_vs_low_q_value")
    res["high_minus_low_d"] = numeric_col(res, "high_minus_low_d")
    for exposure in MAIN_EXPOSURES:
        sub = res[res["exposure"] == exposure].copy()
        if sub.empty:
            continue
        inc = sub[sub["standardized_beta"] > 0].sort_values(["q_value", "adjusted_p", "standardized_beta"], ascending=[True, True, False], na_position="last").head(top_n)
        dec = sub[sub["standardized_beta"] < 0].sort_values(["q_value", "adjusted_p", "standardized_beta"], ascending=[True, True, True], na_position="last").head(top_n)
        for direction, chunk in [("increases_with_exposure", inc), ("decreases_with_exposure", dec)]:
            for rank, (_, row) in enumerate(chunk.iterrows(), start=1):
                rows.append({
                    "exposure": exposure,
                    "direction": direction,
                    "rank": rank,
                    "outcome": row["outcome"],
                    "label": compact_label(row["outcome"]),
                    "standardized_beta": row["standardized_beta"],
                    "adjusted_p": row["adjusted_p"],
                    "q_value": row["q_value"],
                    "high_minus_low_d": row["high_minus_low_d"],
                    "high_vs_low_q": row["high_vs_low_q"],
                    "n": row.get("n", np.nan),
                    "r2": row.get("r2", np.nan),
                })
    return pd.DataFrame(rows, columns=keep_cols)


def find_opposite_axis_candidates(results: pd.DataFrame, top_n: int = OPPOSITE_AXIS_TOP_N) -> pd.DataFrame:
    if results.empty:
        return pd.DataFrame()
    res = results.copy()
    res["standardized_beta"] = numeric_col(res, "exposure_standardized_beta")
    res["q_value"] = numeric_col(res, "exposure_q_value")
    res["adjusted_p"] = numeric_col(res, "exposure_p")
    rows = []
    for outcome, sub in res.groupby("outcome"):
        sub = sub[sub["exposure"].isin(MAIN_EXPOSURES)].copy()
        sub = sub[np.isfinite(sub["standardized_beta"])]
        if sub.empty:
            continue
        pos = sub[sub["standardized_beta"] > 0]
        neg = sub[sub["standardized_beta"] < 0]
        if pos.empty or neg.empty:
            continue
        strongest_pos = pos.loc[pos["standardized_beta"].idxmax()]
        strongest_neg = neg.loc[neg["standardized_beta"].idxmin()]
        rows.append({
            "outcome": outcome,
            "label": compact_label(outcome),
            "positive_axes": ", ".join(pos.sort_values("standardized_beta", ascending=False)["exposure"].map(pretty_label).tolist()),
            "negative_axes": ", ".join(neg.sort_values("standardized_beta")["exposure"].map(pretty_label).tolist()),
            "strongest_positive_axis": strongest_pos["exposure"],
            "strongest_positive_beta": strongest_pos["standardized_beta"],
            "strongest_positive_q": strongest_pos["q_value"],
            "strongest_negative_axis": strongest_neg["exposure"],
            "strongest_negative_beta": strongest_neg["standardized_beta"],
            "strongest_negative_q": strongest_neg["q_value"],
            "axis_beta_range": strongest_pos["standardized_beta"] - strongest_neg["standardized_beta"],
            "best_q_any_axis": sub["q_value"].min(),
            "best_p_any_axis": sub["adjusted_p"].min(),
            "n_axes_tested": sub["exposure"].nunique(),
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["best_q_any_axis", "axis_beta_range", "best_p_any_axis"], ascending=[True, False, True], na_position="last").head(top_n)


def plot_same_outcome_across_axes(results: pd.DataFrame, outcome: str):
    sub = results[(results["outcome"] == outcome) & (results["exposure"].isin(MAIN_EXPOSURES))].copy()
    sub = sub.set_index("exposure").reindex(MAIN_EXPOSURES).reset_index()
    sub["standardized_beta"] = numeric_col(sub, "exposure_standardized_beta")
    sub["q_value"] = numeric_col(sub, "exposure_q_value")
    sub["high_minus_low_d"] = numeric_col(sub, "high_minus_low_d")

    fig, ax = plt.subplots(figsize=(9.8, 5.1), dpi=140)
    x = np.arange(len(sub))
    colors = np.where(sub["standardized_beta"] >= 0, "#b2182b", "#2166ac")
    ax.bar(x, sub["standardized_beta"], color=colors, edgecolor="black", alpha=0.82)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels([pretty_label(e) for e in sub["exposure"]], rotation=30, ha="right")
    for i, row in sub.iterrows():
        beta = row["standardized_beta"]
        if not np.isfinite(beta):
            continue
        p_val = row.get("exposure_p", np.nan)
        q_txt = f"q={row['q_value']:.2g}" if np.isfinite(row["q_value"]) else "q=NA"
        p_txt = f"p={p_val:.2g}" if np.isfinite(p_val) else "p=NA"
        d_txt = f"d={row['high_minus_low_d']:.2g}" if np.isfinite(row["high_minus_low_d"]) else "d=NA"
        va = "bottom" if beta >= 0 else "top"
        offset = 0.015 if beta >= 0 else -0.015
        ax.text(i, beta + offset, f"{p_txt}\n{q_txt}\n{d_txt}", ha="center", va=va, fontsize=7)
    setup_axis(ax, f"Same oral feature across dietary nitrogen axes: {compact_label(outcome)}", "Dietary nitrogen axis", "Adjusted standardized beta")
    fig.tight_layout()
    return fig, ax


def plot_demographic_distribution_violin(base_df: pd.DataFrame, demographic: str, outcome: str, exposure_context: str | None = None):
    df = base_df.copy()
    if demographic == "sex":
        group_col = "sex_group"
        order = [g for g in ["male", "female"] if g in set(df[group_col].dropna())]
        colors = {"male": "#2166ac", "female": "#b2182b"}
        title = f"Sex difference in oral ecology: {compact_label(outcome)}"
        xlab = "Sex"
    elif demographic == "age":
        group_col = "age_tertile_for_plot"
        if "age" not in df.columns:
            return None, None
        age_vals = pd.to_numeric(df["age"], errors="coerce")
        df[group_col] = pd.qcut(age_vals.rank(method="first"), q=3, labels=["younger", "middle", "older"])
        order = ["younger", "middle", "older"]
        colors = {"younger": "#2166ac", "middle": "#f4a582", "older": "#b2182b"}
        title = f"Age-tertile difference in oral ecology: {compact_label(outcome)}"
        xlab = "Age tertile"
    else:
        return None, None

    vals = [pd.to_numeric(df.loc[df[group_col] == g, outcome], errors="coerce").dropna().to_numpy(float) for g in order]
    safe_vals = [v if len(v) else np.array([np.nan]) for v in vals]
    demo_name = "sex" if demographic == "sex" else "age"
    demo_row = demographic_oral_results[(demographic_oral_results["demographic"] == demo_name) & (demographic_oral_results["outcome"] == outcome)].copy()
    if exposure_context is not None and "exposure_context" in demo_row.columns:
        matched = demo_row[demo_row["exposure_context"] == exposure_context]
        if len(matched):
            demo_row = matched
    demo_row = demo_row.iloc[0].to_dict() if len(demo_row) else {}
    exclusion_note = "age excluded from covariates" if demographic == "age" else "sex excluded from covariates"

    fig, ax = plt.subplots(figsize=(7.4, 5.0), dpi=140)
    pos = np.arange(1, len(order) + 1)
    parts = ax.violinplot(safe_vals, positions=pos, widths=0.72, showmeans=False, showextrema=False, showmedians=False)
    for body, group in zip(parts["bodies"], order):
        body.set_facecolor(colors.get(group, "#999999")); body.set_edgecolor("black"); body.set_alpha(0.45)
    box = ax.boxplot(safe_vals, positions=pos, patch_artist=True, widths=0.32, showfliers=False)
    for patch, group in zip(box["boxes"], order):
        patch.set_facecolor(colors.get(group, "#999999")); patch.set_edgecolor("black"); patch.set_alpha(0.78)
    for element in ["whiskers", "caps", "medians"]:
        for item in box[element]:
            item.set_color("black")
    ax.set_xticks(pos)
    ax.set_xticklabels([f"{g}\nn={len(v)}" for g, v in zip(order, vals)])
    q = demo_row.get("q_value", np.nan)
    p = demo_row.get("p_value", np.nan)
    beta = demo_row.get("standardized_beta", np.nan)
    ax.text(0.5, 0.98, f"adjusted beta={beta:.2g}; p={p:.2g}; q={q:.2g}\n{exclusion_note}", transform=ax.transAxes, ha="center", va="top", fontsize=9, bbox=dict(facecolor="white", edgecolor="black", boxstyle="square,pad=0.25"))
    setup_axis(ax, title, xlab, compact_label(outcome))
    fig.tight_layout()
    return fig, ax


print("Ranked taxa/features increasing and decreasing under each dietary nitrogen axis")
directional_taxa_lists = make_directional_taxa_lists(ecology_results)
display(directional_taxa_lists)

for exposure in MAIN_EXPOSURES:
    print("\n" + "=" * 88)
    print(f"{pretty_label(exposure)}: top increases")
    display(directional_taxa_lists[(directional_taxa_lists["exposure"] == exposure) & (directional_taxa_lists["direction"] == "increases_with_exposure")].head(DIRECTION_LIST_TOP_N))
    print(f"{pretty_label(exposure)}: top decreases")
    display(directional_taxa_lists[(directional_taxa_lists["exposure"] == exposure) & (directional_taxa_lists["direction"] == "decreases_with_exposure")].head(DIRECTION_LIST_TOP_N))

print("\nCandidate oral features that increase under one nitrogen axis and decrease under another")
opposite_axis_candidates = find_opposite_axis_candidates(ecology_results)
display(opposite_axis_candidates)

print("Paper candidate plots: same bacterium/feature across all dietary nitrogen axes")
for outcome in opposite_axis_candidates["outcome"].head(6).tolist():
    fig, ax = plot_same_outcome_across_axes(ecology_results, outcome)
    plt.show()
    # fig.savefig(OUTPUT_DIR / f"fig_axis_opposition_{sanitize_feature_name(outcome)}.png", dpi=300, bbox_inches="tight")

print("Publication violin plots for selected diet-to-oral associations")
violin_candidates = pd.concat([
    ecology_results.sort_values(["exposure_q_value", "exposure_p"], na_position="last").head(8),
    ecology_results.assign(abs_beta=ecology_results["exposure_standardized_beta"].abs()).sort_values(["abs_beta", "exposure_q_value"], ascending=[False, True], na_position="last").head(8),
]).drop_duplicates(["exposure", "outcome"]).head(12)
display(violin_candidates[["exposure", "outcome", "exposure_standardized_beta", "exposure_p", "exposure_q_value", "high_minus_low_d", "high_vs_low_q_value"]])
for _, row in violin_candidates.iterrows():
    fig, ax = plot_group_violin(analysis_tables[row["exposure"]], row["exposure"], row["outcome"])
    plt.show()
    # fig.savefig(OUTPUT_DIR / f"fig_violin_{row['exposure']}_{sanitize_feature_name(row['outcome'])}.png", dpi=300, bbox_inches="tight")

print("Age and sex oral ecology examples for manuscript figures")
base_plot_df = analysis_tables[MAIN_EXPOSURES[0]].copy()
age_examples = demographic_oral_results[demographic_oral_results["demographic"] == "age"].sort_values(["q_value", "p_value"], na_position="last").head(4)
sex_examples = demographic_oral_results[demographic_oral_results["demographic"] == "sex"].sort_values(["q_value", "p_value"], na_position="last").head(4)
display(pd.concat([age_examples.assign(plot_group="age_examples"), sex_examples.assign(plot_group="sex_examples")], ignore_index=True))
for outcome in age_examples["outcome"].drop_duplicates().tolist()[:3]:
    fig, ax = plot_demographic_distribution_violin(base_plot_df, "age", outcome, exposure_context=MAIN_EXPOSURES[0])
    if fig is not None:
        plt.show()
        # fig.savefig(OUTPUT_DIR / f"fig_age_violin_{sanitize_feature_name(outcome)}.png", dpi=300, bbox_inches="tight")
for outcome in sex_examples["outcome"].drop_duplicates().tolist()[:3]:
    fig, ax = plot_demographic_distribution_violin(base_plot_df, "sex", outcome, exposure_context=MAIN_EXPOSURES[0])
    if fig is not None:
        plt.show()
        # fig.savefig(OUTPUT_DIR / f"fig_sex_violin_{sanitize_feature_name(outcome)}.png", dpi=300, bbox_inches="tight")

paper_writer_summary = {
    "main_tests": int(ecology_results["exposure_p"].notna().sum()),
    "main_fdr_significant_q_lt_0_05": int((ecology_results["exposure_q_value"] < 0.05).sum()),
    "main_fdr_suggestive_q_lt_0_10": int((ecology_results["exposure_q_value"] < 0.10).sum()),
    "strongest_diet_to_oral_rows": ecology_results.sort_values(["exposure_q_value", "exposure_p"], na_position="last").head(12)[["exposure", "outcome", "exposure_standardized_beta", "exposure_p", "exposure_q_value", "high_minus_low_d"]],
    "opposite_axis_candidates": opposite_axis_candidates,
    "age_oral_examples": age_examples[["demographic", "outcome", "standardized_beta", "p_value", "q_value"]],
    "sex_oral_examples": sex_examples[["demographic", "outcome", "standardized_beta", "p_value", "q_value"]],
}

print("Paper-writing summary: copy/screenshot these objects for manuscript drafting")
print("Main tests:", paper_writer_summary["main_tests"])
print("FDR q<0.05:", paper_writer_summary["main_fdr_significant_q_lt_0_05"])
print("FDR q<0.10:", paper_writer_summary["main_fdr_suggestive_q_lt_0_10"])
print("Strongest diet-to-oral rows")
display(paper_writer_summary["strongest_diet_to_oral_rows"])
print("Opposite-axis candidates")
display(paper_writer_summary["opposite_axis_candidates"])
print("Age oral examples")
display(paper_writer_summary["age_oral_examples"])
print("Sex oral examples")
display(paper_writer_summary["sex_oral_examples"])


## Publication Tables


In [ ]:
main_ecology_table = ecology_results.sort_values(["exposure_q_value", "exposure_p"], na_position="last").copy()
main_demographic_oral_table = demographic_oral_results.sort_values(["q_value", "p_value"], na_position="last").copy()
main_nitrite_nitroso_table = nitrite_nitroso_results.sort_values(["indirect_q_value", "indirect_sobel_p", "abs_indirect_beta"], ascending=[True, True, False], na_position="last").copy()
main_mediation_table = mediation_results.sort_values(["indirect_q_value", "indirect_sobel_p", "abs_indirect_beta"], ascending=[True, True, False], na_position="last").copy()

print("Table 1. Prespecified nitrate/nitrite/nitroso diet-to-oral ecology associations")
display(main_ecology_table[["exposure", "outcome", "metric", "n", "exposure_beta", "exposure_standardized_beta", "exposure_p", "exposure_q_value", "high_minus_low_d", "high_vs_low_p", "high_vs_low_q_value", "r2"]].head(80))

print("Table 2. Age and sex differences in oral ecology")
display(main_demographic_oral_table[[c for c in ["demographic", "outcome", "n", "beta", "se", "standardized_beta", "standardized_se", "p_value", "q_value", "r2"] if c in main_demographic_oral_table.columns]].drop_duplicates().head(80))

print("Table 3. Processed nitrite -> nitroso axis -> oral ecology pathway results")
display(main_nitrite_nitroso_table[["upstream_exposure", "mediator_exposure", "outcome", "n", "a_processed_to_nitroso_beta", "a_q_value", "b_nitroso_to_outcome_beta_adjusted", "b_q_value", "total_processed_to_outcome_beta", "total_q_value", "direct_processed_to_outcome_beta_after_nitroso", "direct_q_value", "indirect_beta_product_ab", "indirect_sobel_p", "indirect_q_value", "fraction_attenuated_after_nitroso"]].head(80))

print("Table 4. Mediation-style age/sex -> diet -> oral ecology results")
display(main_mediation_table[["upstream", "exposure", "outcome", "n", "a_upstream_to_diet_beta", "a_q_value", "b_diet_to_outcome_beta_adjusted", "b_q_value", "total_upstream_to_outcome_beta", "total_upstream_q_value", "direct_upstream_to_outcome_beta_after_diet", "direct_upstream_q_value", "indirect_beta_product_ab", "indirect_sobel_p", "indirect_q_value", "fraction_attenuated_after_diet"]].head(80))

print("Table 5. Age/sex diet differences aligned with oral ecology differences; exploratory, not mediation and not within-group difference")
display(demographic_diet_oral_alignment.head(100))

print("Table 6. Diet-stratified age/sex differences in oral ecology within low/mid/high exposure groups")
display(diet_stratified_demographic_results.sort_values(["q_value", "p_value", "abs_standardized_beta"], ascending=[True, True, False], na_position="last")[["exposure", "exposure_group", "demographic", "outcome", "n", "standardized_beta", "p_value", "q_value", "covariate_exclusion"]].head(120))

print("Interpretation guardrail: main paper should claim distinct oral ecology signatures of nitrate/nitrite/nitroso axes. Processed nitrite/nitroso and age/sex mediation-style sections are observational pathway evidence, not causal mediation proof.")


## 7. Supplementary Transparency


In [ ]:
print("=== Exposure construction ===")
print("KG_MODE:", KG_MODE)
print("KG_MAX_HOPS:", KG_MAX_HOPS)
print("PRIMARY_EXPOSURE_METRIC:", PRIMARY_EXPOSURE_METRIC)
print("SECONDARY_EXPOSURE_METRIC:", SECONDARY_EXPOSURE_METRIC)
print("Reliable logging filter:", PRIMARY_LOGGING_QUALITY)
print("Exposure specs:")
display(pd.DataFrame([{ "exposure": k, **v } for k, v in EXPOSURE_SPECS.items()]))
print("Exposure summary:")
display(exposure_summary)

print("=== Oral microbiome features ===")
print("Relevant taxa patterns:")
display(pd.DataFrame([{"taxon_group": k, "regex": v} for k, v in RELEVANT_TAXA_PATTERNS.items()]))
print("Main oral ecology outcomes, small FDR family:")
display(pd.DataFrame({"outcome": MAIN_ORAL_OUTCOME_COLS}))
print("All initially selected oral outcomes:")
display(pd.DataFrame({"outcome": ORAL_OUTCOME_COLS}))

print("=== Confounders and covariates ===")
print("Base covariates requested:", BASE_COVARIATES)
print("Logging covariates requested:", LOGGING_COVARIATES)
print("Covariates used:", confounder_cols)
print("Smoking columns searched:", SMOKING_COLUMNS)
print("Alcohol columns searched:", ALCOHOL_COLUMNS)
display(confounder_source_summary)

print("=== Statistical thresholds ===")
print("MIN_GROUP_N:", MIN_GROUP_N)
print("FDR_ALPHA:", FDR_ALPHA)
print("Main FDR family: exposures x prespecified oral ecology outcomes")
print("Demographic oral FDR family: age/sex x prespecified oral ecology outcomes")
print("Processed nitrite/nitroso pathway FDR family: processed_nitrite -> nitroso_axis -> prespecified oral ecology outcomes")
print("Supplementary species FDR family: exposures x all species-level MetaPhlAn features")
print("P adjustment: Benjamini-Hochberg within result family")
print("Effect direction: positive exposure beta = higher oral outcome with higher log1p exposure")
print("Causal language: avoid; mediation/pathway sections are observational attenuation or pathway-consistency evidence.")
print("Nitroso pathway guardrail: nitroso_axis is KG-derived dietary reconstruction, not measured circulating or gastric N-nitroso compounds.")


## 8. Supplementary Analysis: All Species-Level MetaPhlAn Features


In [ ]:
def load_species_wide_for_supplement() -> pd.DataFrame:
    primary, pl = load_pheno_main_table("oral_microbiome", "oral_microbiome", return_loader=True)
    dataset_dir = phenoloader_dataset_dir(pl)
    bulk, path = load_optional_bulk_table(primary, r"^metaphlan_abundance_species_parquet$", dataset_dir)
    if bulk is None:
        raise RuntimeError("Could not load MetaPhlAn species bulk table for supplementary all-species analysis.")
    wide = normalize_wide_microbiome_table(bulk, "supp_species")
    if wide is None:
        raise RuntimeError("Could not normalize species bulk table for supplementary all-species analysis.")
    print("Supplementary species wide table:", wide.shape, "from", path)
    return wide


species_wide = load_species_wide_for_supplement()
species_cols = [c for c in species_wide.select_dtypes(include="number").columns if c != ID_COL]
print("All species-level features tested:", len(species_cols))

supp_rows = []
for exposure_name in MAIN_EXPOSURES:
    df = analysis_tables[exposure_name].merge(species_wide, on=ID_COL, how="inner")
    print(f"Supplementary all-species tests for {exposure_name}: rows={df.shape[0]}, species={len(species_cols)}", flush=True)
    for outcome in species_cols:
        supp_rows.append(exposure_oral_model(df, exposure_name, outcome, confounder_cols))

supplementary_species_results = pd.DataFrame(supp_rows)
supplementary_species_results["exposure_q_value_all_species"] = p_adjust_bh(supplementary_species_results["exposure_p"])
supplementary_species_results["high_vs_low_q_value_all_species"] = p_adjust_bh(supplementary_species_results["high_vs_low_p"])

print("Supplementary all-species adjusted exposure results, large FDR family")
display(supplementary_species_results.sort_values(["exposure_q_value_all_species", "exposure_p"], na_position="last").head(100))

print("Supplementary FDR denominator:", supplementary_species_results["exposure_p"].notna().sum())
# supplementary_species_results.to_csv(OUTPUT_DIR / "supplementary_all_species_exposure_results.csv", index=False)
